# 03. Heuristiken bauen: gierig, aber mit Plan

In der 2. Runde gibt es oft keinen Algorithmus, der das Optimum findet. Du baust eine **Heuristik**: ein Verfahren, das schnell eine **gute**, aber nicht unbedingt die beste Lösung liefert
(Begriff aus [Notebook 16](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/16_heuristiken.ipynb) von learn_bwinf1). Die einfachste Sorte ist die **konstruktive Heuristik**: Sie baut die Lösung **Schritt für Schritt** auf und nimmt nichts zurück,
also ein gieriges Verfahren wie in [Notebook 12](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/12_greedy.ipynb). Dort konntest du die Richtigkeit oft **beweisen** (Austauschargument). Hier gibt es meistens **keinen Beweis**, deshalb **misst** du.

In diesem Notebook lernst du:

- eine konstruktive Heuristik in **zwei Teile** zu zerlegen: die **Reihenfolge-Regel** (wer kommt zuerst dran?) und die **Platzierungs-Regel** (wohin kommt er?). Die Regel übergibst du als Funktion, wie in [Notebook 01](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf2/01_pruefen_und_schranken.ipynb),
- **mehrere Regeln** zu entwerfen und sie an **vielen Zufallsfällen** als **Tabelle** zu vergleichen (Mittelwert, schlechtester Fall, wie oft optimal), mit dem Optimum kleiner Fälle als Maßstab,
- zu jeder Regel ein **Gegenbeispiel** zu finden, und zwar **automatisch** durch Vergleich mit Brute Force,
- **Kandidatenstellen** zu benutzen: Statt jede Position zu probieren, prüft man nur die Stellen, an denen sich etwas ändert,
- Regeln zu **kombinieren** („nimm das beste Ergebnis von k Regeln“) und **ehrlich** zu sagen, dass das k-mal so viel kostet,
- das alles an einer echten Aufgabe anzuwenden und mit einem **Solver** (MiniZinc) die Güte der Regeln zu messen.

Die neutralen Beispiele sind **Koffer packen** (Rucksack mit Wert und Gewicht) und **Plakate an eine Pinnwand hängen**. Die Wettbewerbs-Ecken sind der Flohmarkt in Langdorf (nach Aufgabe „Flohmarkt in Langdorf“)
und die Siedler (nach Aufgabe „Die Siedler“, für Schnelle).

## 1. Eine konstruktive Heuristik besteht aus zwei Regeln

Dein Koffer trägt **höchstens 12 kg**. Jeder Gegenstand hat einen **Wert** und ein **Gewicht**. Du willst möglichst viel Wert mitnehmen (das **Rucksackproblem**, das du aus
[Notebook 3](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/03_schranken_symmetrien.ipynb) und [Notebook 11](https://ktheu.github.io/bwinfN/notebooks/index.html?path=learn_bwinf1/11_tabellen_dp.ipynb) kennst).

```python
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12
```

Eine **konstruktive Heuristik** hat hier zwei Bausteine:

1. **Reihenfolge-Regel:** In welcher Reihenfolge werden die Gegenstände betrachtet? Man sortiert sie, am Anfang stehen die „besten“.
2. **Platzierungs-Regel:** Was passiert mit dem Gegenstand, der gerade dran ist? Hier einfach: Passt er noch in den Koffer, kommt er hinein, sonst bleibt er draußen. (Bei den Plakaten in Abschnitt 4 wird auch die Frage „**wohin**?“ wichtig.)

Wir trennen die beiden Regeln im Programm: Die **Reihenfolge-Regel** ist eine **Funktion `regel(gegenstand)`**, die eine Zahl oder ein Tupel zurückgibt, und `sorted(..., key=...)` stellt den Gegenstand mit dem **kleinsten** Schlüssel nach vorn.
Dass man eine Funktion als Argument übergibt, kennst du aus Notebook 01. Wie dort prüft ein **Schiedsrichter** jede Lösung:

In [ ]:
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

print(pruefe_koffer(gegenstaende, kapazitaet, [0, 1, 4]), wert_von(gegenstaende, [0, 1, 4]))
print(pruefe_koffer(gegenstaende, kapazitaet, [0, 1, 2]))
print(pruefe_koffer(gegenstaende, kapazitaet, [0, 0]))

#### Aufgabe 1.1

Drei sinnvolle Reihenfolge-Regeln stehen schon da (`schluessel_wert`, `schluessel_leicht`, `schluessel_wert_pro_kg`). Schreibe `packe(gegenstaende, kapazitaet, regel)`:
Die Gegenstände werden in der Reihenfolge der **Schlüssel** betrachtet (die Sortierzeile steht schon da, `regel` ist eine Funktion, die auf einen Gegenstand `(Wert, Gewicht)` angewendet wird).
Jeder Gegenstand, der **noch passt**, kommt in den Koffer. Die Funktion gibt die Liste der eingepackten Gegenstandsnummern zurück, **in der Reihenfolge des Einpackens**.

**Erwartete Ausgabe:**
```
Wert           [0, 1, 4] Wert 26 (ok)
leicht zuerst  [6, 4, 1, 2] Wert 25 (ok)
Wert pro kg    [1, 6, 4, 2] Wert 25 (ok)
```

In [ ]:
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        # dein Code: passt Gegenstand i noch? Dann einpacken und die freie Kapazität verkleinern
        pass
    return auswahl

REGELN = [('Wert', schluessel_wert), ('leicht zuerst', schluessel_leicht), ('Wert pro kg', schluessel_wert_pro_kg)]
for name, regel in REGELN:
    auswahl = packe(gegenstaende, kapazitaet, regel)
    print(f'{name:14} {auswahl} Wert {wert_von(gegenstaende, auswahl)} ({pruefe_koffer(gegenstaende, kapazitaet, auswahl)})')

<details>
<summary>Tipp anzeigen</summary>

In der Schleife: `if gegenstaende[i][1] <= frei:` dann `auswahl.append(i)` und `frei = frei - gegenstaende[i][1]`. Nicht passende Gegenstände einfach überspringen, es gibt kein `break`: Ein späterer, leichterer Gegenstand kann noch passen.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

REGELN = [('Wert', schluessel_wert), ('leicht zuerst', schluessel_leicht), ('Wert pro kg', schluessel_wert_pro_kg)]
for name, regel in REGELN:
    auswahl = packe(gegenstaende, kapazitaet, regel)
    print(f'{name:14} {auswahl} Wert {wert_von(gegenstaende, auswahl)} ({pruefe_koffer(gegenstaende, kapazitaet, auswahl)})')
```

Bei „Wert“ kommen zuerst der Gegenstand 0 (12 Euro, 7 kg) und der Gegenstand 1 (9, 3 kg). Dann ist das Gewicht 10, und es passt noch der Gegenstand 4 (5, 2 kg): Wert 26. „Leicht zuerst“ beginnt mit dem 1 kg schweren Gegenstand 6 und füllt den Koffer mit vier leichten Gegenständen auf (10 kg, Wert 25).
„Wert pro kg“ beginnt mit Gegenstand 1 (3 Euro pro kg, Gegenstand 6 hat ebenfalls 3) und endet ebenfalls bei 25. **Jede der drei Regeln liefert ein anderes Ergebnis**, und gerade die „schlaueste“ gewinnt hier nicht.

</details>

#### Frage 1.2

1. Folgt aus diesem Beispiel, dass die Regel „Wert“ die beste ist? Und ist sie hier **optimal**?
2. „Leicht zuerst“ und „Wert pro kg“ liefern **verschiedene** Auswahlen mit demselben Wert 25. Ist eine von beiden falsch?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Nein. Ein einzelnes Beispiel zeigt nur, dass **diese** Regel **hier** gewinnt. Ob 26 **optimal** ist, wissen wir noch nicht: Dafür bräuchten wir eine Schranke oder das Optimum zum Vergleichen (Abschnitt 2 rechnet es aus: Es ist optimal).
2. Nein, beide sind erlaubt. Es sind **Heuristiken**, keine Beweise: Zwei verschiedene erlaubte Lösungen können gleich gut sein, und der Schiedsrichter sagt für beide „ok“. Wichtig ist nur, dass die Lösung **erlaubt** ist und wie **gut** sie ist.

</details>

## 2. Regeln an vielen Fällen messen

Eine Regel bewertet man nicht an einem Beispiel, sondern an **vielen**. Wir brauchen dafür drei Dinge:

- **Zufallsfälle** mit festem Startwert, damit die Zahlen reproduzierbar sind (Zufall nur mit `random.Random(startwert)`),
- den **Maßstab**: das **Optimum**. Für kleine Fälle berechnet es die Rucksack-Tabelle aus Notebook 11, für ganz kleine auch das Durchprobieren aller Teilmengen (Notebook 1),
- eine **Kennzahl**: die **Güte** = (Wert der Heuristik) / (Optimum), in Prozent.

Der Fall `zufallsfall(startwert, n)` hat `n` Gegenstände mit Wert 1 bis 20 und Gewicht 2 bis 10. Der Koffer trägt ein Drittel des Gesamtgewichts, also passt etwa ein Drittel der Gegenstände hinein.

In [ ]:
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

g1, k1 = zufallsfall(1, 8)
print(g1, k1)
print(exakt(g1, k1), exakt_probieren(g1, k1))

`exakt` und `exakt_probieren` liefern denselben Wert. Für unsere Fälle ist `exakt` viel schneller, deshalb nehmen wir es als Maßstab. (Beim Erzeugen dieses Notebooks wurde das für 200 Fälle geprüft.)

Jetzt die Messung. Ein **Verfahren** ist hier eine Funktion `verfahren(gegenstaende, kapazitaet)`, die den **Wert** der gefundenen Lösung zurückgibt. So können wir später auch kombinierte Verfahren in dieselbe Messung stecken.

#### Aufgabe 2.1

Schreibe `messe(verfahren, anzahl, n)`: Für die Startwerte `1` bis `anzahl` wird `zufallsfall(startwert, n)` erzeugt, das Verfahren aufgerufen und mit dem Optimum (`exakt`) verglichen.
Die Funktion gibt ein Tripel zurück: die **mittlere Güte** in Prozent, die **schlechteste Güte** in Prozent und **in wie vielen Fällen die Güte genau 100 %** ist (der Wert ist dann gleich dem Optimum).
Die Güte eines Falls ist `wert / optimum` (das Optimum ist hier immer größer als 0).

**Erwartete Ausgabe:**
```
Verfahren         Mittel  schlechtester Fall  optimal
Wert              93.5 %          56.8 %     84 von 200
leicht zuerst     82.1 %          43.5 %     15 von 200
Wert pro kg       98.7 %          85.3 %    140 von 200
```

In [ ]:
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

def verfahren_wert(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert))


def verfahren_leicht(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_leicht))


def verfahren_wert_pro_kg(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert_pro_kg))

def messe(verfahren, anzahl, n):
    # dein Code
    return 0.0, 0.0, 0


VERFAHREN = [('Wert', verfahren_wert), ('leicht zuerst', verfahren_leicht), ('Wert pro kg', verfahren_wert_pro_kg)]
print('Verfahren         Mittel  schlechtester Fall  optimal')
for name, verfahren in VERFAHREN:
    mittel, schlecht, optimal = messe(verfahren, 200, 12)
    print(f'{name:14} {mittel:7.1f} % {schlecht:13.1f} % {optimal:6d} von 200')

<details>
<summary>Tipp anzeigen</summary>

Eine Schleife `for startwert in range(1, anzahl + 1):`, darin `g, kap = zufallsfall(startwert, n)` und `gute = verfahren(g, kap) / exakt(g, kap)`.
Summe für den Mittelwert, `min` für den schlechtesten Fall, ein Zähler für `gute == 1`. Am Ende mit 100 multiplizieren. Der Mittelwert ist `summe / anzahl`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

def verfahren_wert(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert))


def verfahren_leicht(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_leicht))


def verfahren_wert_pro_kg(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert_pro_kg))

def messe(verfahren, anzahl, n):
    summe = 0.0
    schlechtester = 1.0
    optimal = 0
    for startwert in range(1, anzahl + 1):
        g, kap = zufallsfall(startwert, n)
        gute = verfahren(g, kap) / exakt(g, kap)
        summe += gute
        schlechtester = min(schlechtester, gute)
        if gute == 1:
            optimal += 1
    return 100 * summe / anzahl, 100 * schlechtester, optimal

VERFAHREN = [('Wert', verfahren_wert), ('leicht zuerst', verfahren_leicht), ('Wert pro kg', verfahren_wert_pro_kg)]
print('Verfahren         Mittel  schlechtester Fall  optimal')
for name, verfahren in VERFAHREN:
    mittel, schlecht, optimal = messe(verfahren, 200, 12)
    print(f'{name:14} {mittel:7.1f} % {schlecht:13.1f} % {optimal:6d} von 200')
```

Die **Wert-Regel** ist im Mittel brauchbar, hat aber einen **schlechten Fall** weit unter dem Mittel. Die **Wert-pro-kg-Regel** ist im Mittel und im schlechtesten Fall klar die beste, aber auch sie ist **nicht immer optimal**.
„Leicht zuerst“ verschenkt Wert: Sie füllt den Koffer mit wertlosem Kleinkram.

</details>

#### Frage 2.2

1. Lies die Tabelle: Welche Regel würdest du für einen Wettbewerb wählen, wenn du nur **eine** nimmst? Begründe mit **zwei** Spalten.
2. „Wert pro kg“ ist in der Tabelle sehr gut. Darf man daraus schließen, dass es **immer** optimal ist, oder dass es **fast immer** höchstens um ein paar Prozent daneben liegt? Was sagt die Spalte „schlechtester Fall“?
3. Warum hat der Mittelwert allein wenig Aussagekraft?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. „Wert pro kg“: Es hat den **höchsten Mittelwert** (98,7 %) und den **besten schlechtesten Fall** (85,3 %, gegen 56,8 % bei „Wert“ und 43,5 % bei „leicht zuerst“). Außerdem ist es in 140 von 200 Fällen optimal.
2. Nein. In 60 von 200 Fällen ist es **nicht** optimal. Das **Mittel** sagt, es liege im Schnitt nur um 1,3 % daneben. Der **schlechteste Fall** zeigt aber, dass es **einzelne** Eingaben gibt, bei denen man 14,7 % verliert.
   Gemessen sind nur diese 200 Fälle mit 12 Gegenständen, eine Garantie ist das nicht.
3. Ein guter Mittelwert kann **viele gute und wenige sehr schlechte** Fälle verstecken. In der Dokumentation nennt man deshalb **immer** auch den schlechtesten Fall (oder einen Quantilwert) und wie oft das Optimum erreicht wird.

</details>

### Zu jeder Regel ein Gegenbeispiel finden, automatisch

Dass eine Regel **nicht immer** optimal ist, zeigt ein **Gegenbeispiel**: ein Fall, in dem die Regel weniger Wert bringt als das Optimum. Von Hand sucht man lange. Dein Computer sucht für dich: Er probiert Zufallsfälle mit nur **4 Gegenständen** (die du bequem von Hand nachrechnest),
bis das Brute Force (`exakt_probieren`) mehr findet als die Regel. So findest du zudem das **kleinste** Gegenbeispiel in Zufallsfällen.

#### Aufgabe 2.3

Schreibe `finde_gegenbeispiel(verfahren, n)`: Probiere die Startwerte `1, 2, 3, ...` mit `zufallsfall(startwert, n)`. Sobald das Verfahren **weniger** liefert als `exakt_probieren`, gibt die Funktion `(startwert, gegenstaende, kapazitaet, wert, optimum)` zurück.
(Für `n = 4` gibt es für alle drei Regeln ein Gegenbeispiel unter den ersten 2000 Startwerten.)

**Erwartete Ausgabe:**
```
Wert: Startwert 16, Gegenstände [(12, 9), (16, 6), (14, 5), (15, 2)], Kapazität 7: Regel 16, Optimum 29
leicht zuerst: Startwert 2, Gegenstände [(2, 3), (3, 7), (6, 6), (9, 5)], Kapazität 7: Regel 2, Optimum 9
Wert pro kg: Startwert 3, Gegenstände [(8, 10), (5, 7), (20, 9), (19, 3)], Kapazität 9: Regel 19, Optimum 20
```

In [ ]:
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

def verfahren_wert(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert))


def verfahren_leicht(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_leicht))


def verfahren_wert_pro_kg(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert_pro_kg))

VERFAHREN = [('Wert', verfahren_wert), ('leicht zuerst', verfahren_leicht), ('Wert pro kg', verfahren_wert_pro_kg)]


def finde_gegenbeispiel(verfahren, n):
    # dein Code
    return None


print('noch nicht fertig')

<details>
<summary>Tipp anzeigen</summary>

Eine `while True:`-Schleife mit einem Zähler `startwert`, der in jedem Durchgang um 1 größer wird. Beim ersten Fall mit `wert < opt` `return` mit allen fünf Werten. Die Schleife endet nur durch das `return`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

def verfahren_wert(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert))


def verfahren_leicht(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_leicht))


def verfahren_wert_pro_kg(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert_pro_kg))

VERFAHREN = [('Wert', verfahren_wert), ('leicht zuerst', verfahren_leicht), ('Wert pro kg', verfahren_wert_pro_kg)]


def finde_gegenbeispiel(verfahren, n):
    startwert = 1
    while True:
        g, kap = zufallsfall(startwert, n)
        wert = verfahren(g, kap)
        opt = exakt_probieren(g, kap)
        if wert < opt:
            return startwert, g, kap, wert, opt
        startwert += 1

for name, verfahren in VERFAHREN:
    startwert, g, kap, wert, opt = finde_gegenbeispiel(verfahren, 4)
    print(f'{name}: Startwert {startwert}, Gegenstände {g}, Kapazität {kap}: Regel {wert}, Optimum {opt}')
```

Die Gegenbeispiele sind winzig. Einmal blockiert ein einzelner Gegenstand, der zwar viel Wert hat, aber so schwer ist, dass nichts anderes mehr hineinpasst. Einmal füllt ein Leichtgewicht den Koffer so, dass der wertvollste Gegenstand nicht mehr passt.

</details>

#### Frage 2.4

Rechne die drei Gegenbeispiele von Hand nach und erkläre jedes in einem Satz:

1. „Wert“: Gegenstände `(12, 9), (16, 6), (14, 5), (15, 2)`, Kapazität 7.
2. „leicht zuerst“: Gegenstände `(2, 3), (3, 7), (6, 6), (9, 5)`, Kapazität 7.
3. „Wert pro kg“: Gegenstände `(8, 10), (5, 7), (20, 9), (19, 3)`, Kapazität 9. Dieses ist besonders lehrreich: Die Regel gilt als die beste, was ist hier schiefgegangen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Der Gegenstand `(16, 6)` hat den größten Wert und kommt zuerst (6 kg), danach ist nur noch 1 kg frei, nichts passt: Wert 16. Besser sind `(14, 5)` und `(15, 2)` zusammen (7 kg, Wert 29).
2. Der leichteste Gegenstand `(2, 3)` kommt zuerst, dann sind 4 kg frei, und kein weiterer passt: Wert 2. Besser ist allein `(9, 5)`.
3. Der Gegenstand `(19, 3)` hat 6,3 Euro pro kg, der Gegenstand `(20, 9)` nur 2,2. Nach `(19, 3)` sind 6 kg frei, `(20, 9)` passt nicht mehr, und die anderen sind schwerer: Wert 19. Optimal ist `(20, 9)` allein.
   Die Regel „Wert pro kg“ **ignoriert, wie voll der Koffer wird**: Sie nimmt das, was am „effizientesten“ ist, und verbaut dabei den Platz für einen großen Gegenstand. Das ist die **Grenze** jeder gierigen Regel: Eine Entscheidung wird nie zurückgenommen.

</details>

### Finde den Fehler: ein Minuszeichen fehlt

Jemand schreibt die Regel „Wert pro kg“ neu und vergisst das Minuszeichen:

In [ ]:
def schluessel_falsch(g):
    return g[0] / g[1]                 # Minus vergessen!


for regel in (schluessel_wert_pro_kg, schluessel_falsch):
    auswahl = packe(gegenstaende, kapazitaet, regel)
    print(auswahl, wert_von(gegenstaende, auswahl), pruefe_koffer(gegenstaende, kapazitaet, auswahl))

[1, 6, 4, 2] 25 ok
[5, 3, 6] 14 ok


#### Frage 2.5

1. Was macht die falsche Regel, und warum meldet der Schiedsrichter trotzdem „ok“?
2. Woran würdest du den Fehler **bemerken**, wenn du nur den Schiedsrichter hättest? Und wenn du die Tabelle aus Abschnitt 2 hättest?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `sorted` stellt den **kleinsten** Schlüssel nach vorn, also kommt jetzt der Gegenstand mit dem **kleinsten** Wert pro kg zuerst: genau die Gegenstände, die man am wenigsten will. Die Auswahl ist trotzdem **erlaubt** (nichts ist zu schwer), also sagt der Schiedsrichter „ok“.
   Er prüft nur die Regeln der Aufgabe, nicht die Qualität.
2. Der Schiedsrichter merkt nichts, **nur die Güte** zeigt es: Hier sind es 14 statt 25 Euro im Koffer-Beispiel, und über 200 Zufallsfälle liegt die Regel im Mittel bei 25,0 % (statt 98,7 %), im schlechtesten Fall bei 4,3 %.
   Eine Tabelle mit mehreren Regeln hilft, weil **ein Ausreißer nach unten** (hier eine Regel, die viel schlechter ist als alle anderen) sofort auffällt.

</details>

## 3. Mehrere Regeln kombinieren: das beste von k

Jede Regel hat **andere schlechte Fälle**: „Wert“ scheitert, wenn ein schwerer Gegenstand den Koffer verstopft, „leicht zuerst“, wenn viel Kleinkram da ist, „Wert pro kg“, wenn ein großer Gegenstand den Platz braucht.
Also probieren wir **alle** Regeln und behalten das **beste Ergebnis**. Das ist eine neue Heuristik, die immer mindestens so gut ist wie jede einzelne.
Weil `packe` die Regel als Funktion bekommt, geht das mit einer Liste von Regeln:

#### Aufgabe 3.1

Schreibe `beste_von(gegenstaende, kapazitaet, regeln)`: Für **jede** Regel der Liste `regeln` wird `packe` aufgerufen, und die Funktion gibt den **größten Wert** zurück.

**Erwartete Ausgabe:**
```
26
25
```

In [ ]:
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

def verfahren_wert(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert))


def verfahren_leicht(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_leicht))


def verfahren_wert_pro_kg(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert_pro_kg))

def beste_von(gegenstaende, kapazitaet, regeln):
    # dein Code
    return 0


alle_drei = [schluessel_wert, schluessel_leicht, schluessel_wert_pro_kg]
print(beste_von(gegenstaende, kapazitaet, alle_drei))
print(beste_von(gegenstaende, kapazitaet, [schluessel_leicht, schluessel_wert_pro_kg]))

<details>
<summary>Tipp anzeigen</summary>

`bester = 0`, dann `for regel in regeln:` und `bester = max(bester, wert_von(gegenstaende, packe(gegenstaende, kapazitaet, regel)))`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
gegenstaende = [(12, 7), (9, 3), (8, 4), (7, 6), (5, 2), (4, 5), (3, 1)]   # (Wert, Gewicht in kg)
kapazitaet = 12

def pruefe_koffer(gegenstaende, kapazitaet, auswahl):
    """auswahl: Liste der Nummern der eingepackten Gegenstände. Gibt 'ok' zurück oder beschreibt den ersten Verstoß."""
    gesehen = set()
    gewicht = 0
    for i in auswahl:
        if i < 0 or i >= len(gegenstaende):
            return f'Gegenstand {i} gibt es nicht'
        if i in gesehen:
            return f'Gegenstand {i} ist doppelt eingepackt'
        gesehen.add(i)
        gewicht += gegenstaende[i][1]
    if gewicht > kapazitaet:
        return f'zu schwer: {gewicht} kg bei höchstens {kapazitaet} kg'
    return 'ok'


def wert_von(gegenstaende, auswahl):
    summe = 0
    for i in auswahl:
        summe += gegenstaende[i][0]
    return summe

def schluessel_wert(g):
    return -g[0]                       # g = (Wert, Gewicht). Kleinster Schlüssel zuerst: der wertvollste Gegenstand kommt vor


def schluessel_leicht(g):
    return g[1]                        # der leichteste zuerst


def schluessel_wert_pro_kg(g):
    return -g[0] / g[1]                # der mit dem größten Wert pro Kilogramm zuerst

def packe(gegenstaende, kapazitaet, regel):
    reihenfolge = sorted(range(len(gegenstaende)), key=lambda i: regel(gegenstaende[i]))
    auswahl = []
    frei = kapazitaet
    for i in reihenfolge:
        if gegenstaende[i][1] <= frei:
            auswahl.append(i)
            frei -= gegenstaende[i][1]
    return auswahl

import random


def zufallsfall(startwert, n):
    zufall = random.Random(startwert)
    gegenstaende = []
    for i in range(n):
        gegenstaende.append((zufall.randint(1, 20), zufall.randint(2, 10)))
    kapazitaet = sum(g[1] for g in gegenstaende) // 3
    return gegenstaende, kapazitaet


def exakt(gegenstaende, kapazitaet):
    """Das Optimum mit der Rucksack-Tabelle: beste[c] ist der größte Wert bei höchstens c kg."""
    beste = [0] * (kapazitaet + 1)
    for wert, gewicht in gegenstaende:
        for c in range(kapazitaet, gewicht - 1, -1):
            beste[c] = max(beste[c], beste[c - gewicht] + wert)
    return beste[kapazitaet]


def exakt_probieren(gegenstaende, kapazitaet):
    """Brute Force: alle Teilmengen (für höchstens etwa 12 Gegenstände)."""
    n = len(gegenstaende)
    bestes = 0
    for maske in range(2 ** n):
        gewicht = 0
        wert = 0
        for i in range(n):
            if maske // 2 ** i % 2 == 1:
                gewicht += gegenstaende[i][1]
                wert += gegenstaende[i][0]
        if gewicht <= kapazitaet and wert > bestes:
            bestes = wert
    return bestes

def verfahren_wert(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert))


def verfahren_leicht(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_leicht))


def verfahren_wert_pro_kg(gegenstaende, kapazitaet):
    return wert_von(gegenstaende, packe(gegenstaende, kapazitaet, schluessel_wert_pro_kg))

def beste_von(gegenstaende, kapazitaet, regeln):
    """Probiert jede Regel der Liste und gibt den größten Wert zurück."""
    bester = 0
    for regel in regeln:
        auswahl = packe(gegenstaende, kapazitaet, regel)
        bester = max(bester, wert_von(gegenstaende, auswahl))
    return bester

alle_drei = [schluessel_wert, schluessel_leicht, schluessel_wert_pro_kg]
print(beste_von(gegenstaende, kapazitaet, alle_drei))
print(beste_von(gegenstaende, kapazitaet, [schluessel_leicht, schluessel_wert_pro_kg]))
```

Mit allen drei Regeln erreicht man 26, das Optimum. Ohne die Regel „Wert“ nur 25.

</details>

Zusätzlich gibt es **Zufallsregeln**: Wer jedem Gegenstand eine Zufallszahl als Schlüssel gibt, probiert eine **zufällige Reihenfolge**. Eine einzelne taugt wenig, aber viele zusammen haben Chancen auf einen Glückstreffer
(mehr zu Zufall und Neustarts im Notebook über lokale Suche). `mache_zufallsregel` **gibt eine Funktion zurück**: Die innere Funktion `regel` merkt sich den Zufallsgenerator `zufall`.

In [ ]:
def beste_von(gegenstaende, kapazitaet, regeln):
    """Probiert jede Regel der Liste und gibt den größten Wert zurück."""
    bester = 0
    for regel in regeln:
        auswahl = packe(gegenstaende, kapazitaet, regel)
        bester = max(bester, wert_von(gegenstaende, auswahl))
    return bester

def mache_zufallsregel(startwert):
    zufall = random.Random(startwert)

    def regel(gegenstand):
        return zufall.random()

    return regel

ALLE_DREI = [schluessel_wert, schluessel_leicht, schluessel_wert_pro_kg]


def verfahren_drei(gegenstaende, kapazitaet):
    return beste_von(gegenstaende, kapazitaet, ALLE_DREI)


def verfahren_drei_und_zufall(gegenstaende, kapazitaet):
    regeln = list(ALLE_DREI)
    for k in range(20):
        regeln.append(mache_zufallsregel(1000 * len(gegenstaende) + k))
    return beste_von(gegenstaende, kapazitaet, regeln)

VERFAHREN = [('Wert', verfahren_wert, 1), ('leicht zuerst', verfahren_leicht, 1), ('Wert pro kg', verfahren_wert_pro_kg, 1),
             ('beste von 3', verfahren_drei, 3), ('beste von 3 + 20 Zufall', verfahren_drei_und_zufall, 23)]
print('Verfahren                 Mittel  schlechtester Fall  optimal      Aufrufe von packe')
for name, verfahren, aufrufe in VERFAHREN:
    mittel, schlecht, optimal = messe(verfahren, 200, 12)
    print(f'{name:22} {mittel:7.1f} % {schlecht:13.1f} % {optimal:6d} von 200 {aufrufe:10d}')

Verfahren                 Mittel  schlechtester Fall  optimal      Aufrufe von packe
Wert                      93.5 %          56.8 %     84 von 200          1
leicht zuerst             82.1 %          43.5 %     15 von 200          1
Wert pro kg               98.7 %          85.3 %    140 von 200          1
beste von 3               99.4 %          91.1 %    168 von 200          3
beste von 3 + 20 Zufall    99.6 %          91.1 %    174 von 200         23


#### Frage 3.2

1. Warum ist „beste von 3“ **nie schlechter** als jede der drei einzelnen Regeln?
2. Die Tabelle zeigt, was die Kombination bringt und was sie **kostet**. Vergleiche „Wert pro kg“, „beste von 3“ und „beste von 3 + 20 Zufall“: Wie viel Güte gewinnt man jeweils, und wie viel mehr Rechenarbeit ist das?
3. Wo würdest du aufhören?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die drei Regeln sind in der Liste enthalten, und `max` nimmt den größten Wert. Es kann also nur besser werden. (Das gilt für **jede** Kombination, die die einzelnen Regeln enthält.)
2. Von „Wert pro kg“ (98,7 %, optimal in 140 Fällen) zu „beste von 3“ (99,4 %, optimal in 168 Fällen): **+0,7 Prozentpunkte** im Mittel für **dreimal** so viel Arbeit
   (3 Aufrufe von `packe`). Die 20 Zufallsregeln bringen weitere **0,2 Prozentpunkte** für **23-mal** so viel Arbeit wie eine einzelne Regel. Der Schlechtestfall steigt von 85,3 % auf 91,1 % und 91,1 %.
3. Das hängt vom Zeitbudget ab. Die **ersten** Regeln, die sich in ihren Schwächen **unterscheiden**, bringen am meisten. Zufall zahlt sich nur aus, wenn man viel Zeit hat und keine bessere Idee. **Jede Regel kostet einen kompletten Durchlauf**:
   k Regeln sind k-mal so langsam, und bei großen Eingaben (wie in der 2. Runde) ist das oft die Grenze. Diese ehrliche Angabe („beste von k Regeln, k-fache Rechenzeit“) gehört in die Dokumentation.

</details>

## 4. Wohin damit? Kandidatenstellen statt aller Stellen

Beim Koffer gab es nur die Frage „einpacken oder nicht“. Oft muss man auch entscheiden, **wo** etwas hinkommt. Ein Beispiel: Plakate an eine **Pinnwand** der Breite `W` und der Höhe `H` (ganze Zahlen). Ein Plakat hat Breite `b` und Höhe `h` und darf sich mit keinem anderen überdecken.
Die Plakate werden in einer Reihenfolge betrachtet, und jedes kommt an die **erste freie Stelle**: Die niedrigste Zeile `y`, in ihr die linkeste Spalte `x`. Findet sich keine Stelle, wird das Plakat nicht aufgehängt. Ziel ist eine möglichst große bedeckte Fläche.
Ein platziertes Plakat steht als `(x, y, b, h)` in einer Liste.

Die Platzierungs-Regel „erste freie Stelle“ kann man auf zwei Arten programmieren. **Alle Stellen** probieren (`stelle_raster`) ist einfach, aber teuer: Es sind bis zu `W * H` Stellen. Das wird bei großen Flächen (ein Flohmarkt mit 1000 Metern!) unbrauchbar.

In [ ]:
import random


def plakat_fall(startwert, n=14, W=40, H=25):
    zufall = random.Random(startwert)
    plakate = []
    for i in range(n):
        plakate.append((zufall.randint(4, 16), zufall.randint(3, 12)))      # (Breite, Höhe)
    return plakate, W, H


def ist_frei(platziert, x, y, b, h, W, H):
    """Passt ein Plakat der Größe b mal h mit der linken unteren Ecke bei (x, y) auf die Wand, ohne ein anderes zu überdecken?"""
    if x + b > W or y + h > H:
        return False
    for px, py, pb, ph in platziert:
        if x < px + pb and px < x + b and y < py + ph and py < y + h:
            return False
    return True


def stelle_raster(platziert, b, h, W, H):
    """Probiert alle Stellen, Zeile für Zeile. Gibt ((x, y) oder None, Zahl der geprüften Stellen) zurück."""
    geprueft = 0
    for y in range(H - h + 1):
        for x in range(W - b + 1):
            geprueft += 1
            if ist_frei(platziert, x, y, b, h, W, H):
                return (x, y), geprueft
    return None, geprueft


def haenge(plakate, W, H, reihenfolge, stelle):
    """stelle ist die Platzierungs-Regel: stelle_raster oder stelle_kandidaten. Gibt (platziert, Zahl der geprüften Stellen) zurück."""
    platziert = []
    geprueft = 0
    for i in reihenfolge:
        b, h = plakate[i]
        pos, z = stelle(platziert, b, h, W, H)
        geprueft += z
        if pos is not None:
            platziert.append((pos[0], pos[1], b, h))
    return platziert, geprueft


def flaeche_von(platziert):
    summe = 0
    for px, py, pb, ph in platziert:
        summe += pb * ph
    return summe

**Kandidatenstellen.** Die erste freie Stelle in der Reihenfolge „niedrigste Zeile, darin linkeste Spalte“ kann nicht irgendwo liegen: Ihre Zeile `y` ist **0 oder die Oberkante** eines platzierten Plakats (sonst könnte man es nach unten schieben),
und ihre Spalte `x` ist **0 oder die rechte Kante** eines platzierten Plakats (sonst könnte man es nach links schieben). Es genügt also, die **Kandidaten** `x in xs` und `y in ys` zu probieren:

- `xs`: die Zahl 0 und `px + pb` für jedes platzierte Plakat,
- `ys`: die Zahl 0 und `py + ph` für jedes platzierte Plakat.

Bei `m` platzierten Plakaten sind das höchstens `(m + 1) * (m + 1)` Stellen statt `W * H`. Die Reihenfolge der Prüfung bleibt: erst die kleineren `y`, dann die kleineren `x`.

#### Aufgabe 4.1

Schreibe `stelle_kandidaten(platziert, b, h, W, H)` mit demselben Ergebnis wie `stelle_raster`, aber nur mit Kandidatenstellen. Sie gibt `((x, y) oder None, Zahl der geprüften Stellen)` zurück.
Sortiere `xs` und `ys` aufsteigend, und entferne Doppelte (`sorted(set(liste))`). Prüfe mit `ist_frei`.

**Erwartete Ausgabe:**
```
((18, 0), 19)
((18, 0), 4)
((6, 8), 79)
((6, 8), 10)
```

In [ ]:
import random


def plakat_fall(startwert, n=14, W=40, H=25):
    zufall = random.Random(startwert)
    plakate = []
    for i in range(n):
        plakate.append((zufall.randint(4, 16), zufall.randint(3, 12)))      # (Breite, Höhe)
    return plakate, W, H


def ist_frei(platziert, x, y, b, h, W, H):
    """Passt ein Plakat der Größe b mal h mit der linken unteren Ecke bei (x, y) auf die Wand, ohne ein anderes zu überdecken?"""
    if x + b > W or y + h > H:
        return False
    for px, py, pb, ph in platziert:
        if x < px + pb and px < x + b and y < py + ph and py < y + h:
            return False
    return True


def stelle_raster(platziert, b, h, W, H):
    """Probiert alle Stellen, Zeile für Zeile. Gibt ((x, y) oder None, Zahl der geprüften Stellen) zurück."""
    geprueft = 0
    for y in range(H - h + 1):
        for x in range(W - b + 1):
            geprueft += 1
            if ist_frei(platziert, x, y, b, h, W, H):
                return (x, y), geprueft
    return None, geprueft


def haenge(plakate, W, H, reihenfolge, stelle):
    """stelle ist die Platzierungs-Regel: stelle_raster oder stelle_kandidaten. Gibt (platziert, Zahl der geprüften Stellen) zurück."""
    platziert = []
    geprueft = 0
    for i in reihenfolge:
        b, h = plakate[i]
        pos, z = stelle(platziert, b, h, W, H)
        geprueft += z
        if pos is not None:
            platziert.append((pos[0], pos[1], b, h))
    return platziert, geprueft


def flaeche_von(platziert):
    summe = 0
    for px, py, pb, ph in platziert:
        summe += pb * ph
    return summe

def stelle_kandidaten(platziert, b, h, W, H):
    xs = [0]
    ys = [0]
    # dein Code: Kandidaten sammeln (rechte und obere Kanten) und sortieren
    geprueft = 0
    # dein Code: für y in ys, für x in xs: zählen und mit ist_frei prüfen
    return None, geprueft

platziert = [(0, 0, 10, 5), (10, 0, 8, 8), (0, 5, 6, 6)]
print(stelle_raster(platziert, 7, 4, 40, 25))
print(stelle_kandidaten(platziert, 7, 4, 40, 25))
print(stelle_raster(platziert, 7, 4, 15, 25))
print(stelle_kandidaten(platziert, 7, 4, 15, 25))

<details>
<summary>Tipp anzeigen</summary>

Für jedes platzierte Plakat `px, py, pb, ph`: `xs.append(px + pb)` und `ys.append(py + ph)`. Danach `xs = sorted(set(xs))` (und ebenso `ys`). Dann zwei verschachtelte Schleifen `for y in ys:` und `for x in xs:` wie in `stelle_raster`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import random


def plakat_fall(startwert, n=14, W=40, H=25):
    zufall = random.Random(startwert)
    plakate = []
    for i in range(n):
        plakate.append((zufall.randint(4, 16), zufall.randint(3, 12)))      # (Breite, Höhe)
    return plakate, W, H


def ist_frei(platziert, x, y, b, h, W, H):
    """Passt ein Plakat der Größe b mal h mit der linken unteren Ecke bei (x, y) auf die Wand, ohne ein anderes zu überdecken?"""
    if x + b > W or y + h > H:
        return False
    for px, py, pb, ph in platziert:
        if x < px + pb and px < x + b and y < py + ph and py < y + h:
            return False
    return True


def stelle_raster(platziert, b, h, W, H):
    """Probiert alle Stellen, Zeile für Zeile. Gibt ((x, y) oder None, Zahl der geprüften Stellen) zurück."""
    geprueft = 0
    for y in range(H - h + 1):
        for x in range(W - b + 1):
            geprueft += 1
            if ist_frei(platziert, x, y, b, h, W, H):
                return (x, y), geprueft
    return None, geprueft


def haenge(plakate, W, H, reihenfolge, stelle):
    """stelle ist die Platzierungs-Regel: stelle_raster oder stelle_kandidaten. Gibt (platziert, Zahl der geprüften Stellen) zurück."""
    platziert = []
    geprueft = 0
    for i in reihenfolge:
        b, h = plakate[i]
        pos, z = stelle(platziert, b, h, W, H)
        geprueft += z
        if pos is not None:
            platziert.append((pos[0], pos[1], b, h))
    return platziert, geprueft


def flaeche_von(platziert):
    summe = 0
    for px, py, pb, ph in platziert:
        summe += pb * ph
    return summe

def stelle_kandidaten(platziert, b, h, W, H):
    xs = [0]
    ys = [0]
    for px, py, pb, ph in platziert:
        xs.append(px + pb)
        ys.append(py + ph)
    xs = sorted(set(xs))
    ys = sorted(set(ys))
    geprueft = 0
    for y in ys:
        for x in xs:
            geprueft += 1
            if ist_frei(platziert, x, y, b, h, W, H):
                return (x, y), geprueft
    return None, geprueft

platziert = [(0, 0, 10, 5), (10, 0, 8, 8), (0, 5, 6, 6)]
print(stelle_raster(platziert, 7, 4, 40, 25))
print(stelle_kandidaten(platziert, 7, 4, 40, 25))
print(stelle_raster(platziert, 7, 4, 15, 25))
print(stelle_kandidaten(platziert, 7, 4, 15, 25))
```

Beide Verfahren finden dieselbe Stelle: `(18, 0)` an der breiten Wand und `(6, 8)` an der schmalen (bei der schmalen Wand hängt das zweite Plakat über den Rand, es zählt nur als Hindernis für die ersten Stellen).
Das Raster musste dafür 19 und 79 Stellen prüfen, die Kandidaten nur 4 und 10: Es gibt in `xs` und in `ys` höchstens je 4 Werte (0 und die Kanten der 3 Plakate), also höchstens 16 Stellen.

</details>

Jetzt prüfen wir, dass beide Platzierungs-Regeln **immer dasselbe Ergebnis** liefern, und zählen die geprüften Stellen. Die Reihenfolge ist „größte Fläche zuerst“.

In [ ]:
def reihenfolge_flaeche(plakate):
    return sorted(range(len(plakate)), key=lambda i: -plakate[i][0] * plakate[i][1])


summe_raster = 0
summe_kandidaten = 0
for startwert in range(1, 31):
    plakate, W, H = plakat_fall(startwert)
    reihenfolge = reihenfolge_flaeche(plakate)
    a, z1 = haenge(plakate, W, H, reihenfolge, stelle_raster)
    b, z2 = haenge(plakate, W, H, reihenfolge, stelle_kandidaten)
    assert a == b, startwert
    summe_raster += z1
    summe_kandidaten += z2
print('30 Fälle, in allen dieselbe Anordnung')
print('geprüfte Stellen mit Raster:', summe_raster)
print('geprüfte Stellen mit Kandidaten:', summe_kandidaten)

30 Fälle, in allen dieselbe Anordnung
geprüfte Stellen mit Raster: 161539
geprüfte Stellen mit Kandidaten: 12220


#### Frage 4.2

1. Mit dem Raster werden in den 30 Fällen 161539 Stellen geprüft, mit den Kandidaten 12220. Um welchen Faktor ist das Kandidatenverfahren sparsamer, ungefähr?
2. Begründe, warum die **erste freie Stelle** (niedrigste Zeile, darin linkeste Spalte) immer unter den Kandidaten ist. Was würde schiefgehen, wenn die Platzierungs-Regel „die Stelle, die den **kleinsten Rest** lässt“ wäre?
3. Wie viele Stellen prüft das Raster höchstens je Plakat, wie viele die Kandidaten, wenn schon `m` Plakate hängen? Wie ändert sich das, wenn die Wand 1000-mal größer wird?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Etwa um den Faktor 13.
2. Liegt die erste freie Stelle bei `(x, y)` und ist `y` weder 0 noch eine Oberkante, dann ist die Zeile `y - 1` für das Plakat an derselben Stelle `x` auch noch frei, also wäre eine **niedrigere** Zeile frei gewesen. Ebenso für `x`: Ist `x` weder 0 noch eine rechte Kante, ist `x - 1` in derselben Zeile frei.
   Die erste freie Stelle liegt also in `xs` mal `ys`. Das gilt, weil die Regel „**nach unten und links** so weit wie möglich“ heißt. Für die Regel „kleinster Rest“ gilt das nicht: Die beste Stelle kann mitten in einer Lücke liegen, und die Kandidaten würden sie verfehlen. Dann braucht man andere Kandidaten.
3. Das Raster höchstens `(W - b + 1) * (H - h + 1)`, also etwa `W * H`. Die Kandidaten höchstens `(m + 1)²`, **unabhängig von der Größe der Wand**. Wird jede Seite der Wand 1000-mal so lang, hat das Raster 10⁶-mal so viele Stellen, die Kandidaten bleiben gleich viele.
   Mit dem Aufruf von `ist_frei` (bis zu `m` Plakate) kommt beim Kandidatenverfahren etwa `m³` Rechenschritte je Plakat heraus, beim Raster `W · H · m`.

</details>

### Mehrere Reihenfolge-Regeln für Plakate vergleichen

Jetzt trennen wir die beiden Regeln wirklich: Die Platzierungs-Regel ist fest (Kandidaten), und wir vergleichen drei **Reihenfolge-Regeln**: nach Fläche, nach Breite, nach Höhe (immer das größte zuerst).
Als **Maßstab** dient eine **obere Schranke** für die bedeckte Fläche: Mehr als die Wand (`W * H`) und mehr als die Summe aller Plakatflächen kann es nie sein. Sie ist nur eine Schranke, das Optimum kann darunter liegen.
Die Spalte „Schranke erreicht“ zählt Fälle, in denen die Fläche **genau** die Schranke ist (dann ist die Lösung **bewiesen optimal**).

In [ ]:
def reihenfolge_nach(plakate, regel):
    return sorted(range(len(plakate)), key=lambda i: regel(plakate[i]))


def schluessel_flaeche(p):
    return -p[0] * p[1]


def schluessel_breite(p):
    return (-p[0], -p[1])


def schluessel_hoehe(p):
    return (-p[1], -p[0])


print('Reihenfolge   Mittel  schlechtester Fall  Schranke erreicht')
for name, regel in [('Fläche', schluessel_flaeche), ('Breite', schluessel_breite), ('Höhe', schluessel_hoehe)]:
    summe = 0.0
    schlecht = 1.0
    erreicht = 0
    for startwert in range(1, 101):
        plakate, W, H = plakat_fall(startwert)
        schranke = min(W * H, sum(b * h for b, h in plakate))
        platziert, z = haenge(plakate, W, H, reihenfolge_nach(plakate, regel), stelle_kandidaten)
        anteil = flaeche_von(platziert) / schranke
        summe += anteil
        schlecht = min(schlecht, anteil)
        if anteil == 1:
            erreicht += 1
    print(f'{name:10} {100 * summe / 100:7.1f} % {100 * schlecht:13.1f} % {erreicht:9d} von 100')

Reihenfolge   Mittel  schlechtester Fall  Schranke erreicht
Fläche        90.0 %          79.9 %        10 von 100
Breite        86.5 %          68.7 %         7 von 100
Höhe          89.7 %          76.0 %        12 von 100


#### Frage 4.3

1. Welche Reihenfolge-Regel liefert im Mittel die **größte** bedeckte Fläche, und wie viel Prozent der Schranke sind das? Und in wie vielen von 100 Fällen ist die Lösung **bewiesen optimal**?
2. Die Tabelle zeigt den Abstand zur **Schranke**, nicht zum Optimum. Darf man sagen „die Regel Fläche liegt im Mittel 10 % neben dem Optimum“?
3. Warum ist es sinnvoll, **Reihenfolge-Regel und Platzierungs-Regel getrennt** zu programmieren?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Die Regel **Fläche** mit 90,0 % der Schranke im Mittel, schlechtester Fall 79,9 %. Bewiesen optimal (Fläche gleich der Schranke) ist die Lösung in 10 von 100 Fällen.
2. Nein. Das Optimum liegt **zwischen** der Lösung und der Schranke. Die Lösung ist also **höchstens** so weit neben dem Optimum, wie sie unter der Schranke liegt, wahrscheinlich weniger. „Im Mittel 10 % unter der Schranke“ ist ehrlich, „10 % neben dem Optimum“ wäre eine Behauptung, die wir nicht bewiesen haben.
3. Man tauscht **eine** Regel aus und sieht sofort, was sie bringt, ohne das Platzieren neu zu schreiben. Und die **Kandidatenstellen** (Platzierung) lassen sich testen, indem man sie gegen das einfache Raster laufen lässt, wie in 4.2, unabhängig davon, in welcher Reihenfolge die Plakate kommen.

</details>

## Wettbewerbs-Ecke 1: Flohmarkt in Langdorf

*Angelehnt an Aufgabe 1 „Flohmarkt in Langdorf“ der 2. Runde des 39. Bundeswettbewerbs Informatik (2020/21).*

Ein Flohmarkt dauert von 8 bis 18 Uhr und liegt an einer Straße der Länge `L` Meter (im Wettbewerb 1000 Meter). Jede **Anmeldung** ist ein Tripel `(von, bis, laenge)`: Der Stand soll von `von` bis `bis` Uhr (volle Stunden)
`laenge` Meter lang sein und kostet 1 Euro je Stunde und Meter, also `(bis - von) * laenge` Euro **Miete**. Zwei Stände, die sich **zeitlich überschneiden**, dürfen sich **räumlich** nicht überschneiden.
Gesucht ist eine **Auswahl** der Anmeldungen und für jede gewählte ein **Standort** (der erste Meter `x`), so dass die **Mieteinnahmen** möglichst groß sind. Ein Stand von 11 bis 16 Uhr und 5 Metern kostet 25 Euro. Ein Stand von 16 bis 18 Uhr darf am selben Ort stehen wie einer, der um 16 Uhr aufhört.

Im Ordner `daten/` liegen drei **echte Beispiele** des Wettbewerbs (`flohmarkt4.txt` mit 7, `flohmarkt5.txt` mit 25 und `flohmarkt6.txt` mit 9 Anmeldungen). Das Format: Die erste Zahl ist die Zahl `n` der Anmeldungen, danach folgen je Anmeldung die drei Zahlen `von bis laenge`.
(Das Lösungsnotebook [Flohmarkt in Langdorf](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/39_FlohmarktInLangdorf/flohmarkt_loesung.ipynb) behandelt auch die großen Beispiele mit 490 bis 735 Anmeldungen.)

Das ist wie bei den Plakaten ein **Packproblem mit Auswahl**, nur dass die Zeit **fest** ist und man in der Ortsrichtung eine Stelle wählt. Die Heuristik wird dieselbe:
**Reihenfolge-Regel** (Anmeldungen sortieren) und **Platzierungs-Regel** („erster Meter, an dem der Stand passt“) mit **Kandidatenstellen**.

#### Aufgabe E1.1

Schreibe `erster_platz(platziert, a, L)`: Sie gibt den **kleinsten Meter `x`** zurück, an dem der Stand `a` auf den Markt der Länge `L` passt, ohne einen Stand aus `platziert` (Liste von `(x, anmeldung)`) zu stören. Gibt es keinen, gibt sie `None` zurück.
Probiere **nur Kandidatenstellen**: den Meter 0 und das **Ende** `xb + b[2]` jedes bereits platzierten Standes `b`, der sich mit `a` **zeitlich überschneidet** (nur der stört ja). Hilfsfunktionen stehen oben (`zeit_ueberlappt`, `stoert`).
Die Kandidaten sortierst du aufsteigend. Wird `x + a[2] > L`, gibt es weiter rechts erst recht keinen Platz mehr.

**Erwartete Ausgabe:**
```
(11, 16, 5) ab Meter 0
(16, 18, 3) ab Meter 0
(10, 14, 4) ab Meter 5
47 ok
None
```

In [ ]:
def lies_markt(dateiname):
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    anmeldungen = []
    for i in range(zahlen[0]):
        anmeldungen.append((zahlen[1 + 3 * i], zahlen[2 + 3 * i], zahlen[3 + 3 * i]))
    return anmeldungen


def miete(a):
    return (a[1] - a[0]) * a[2]


def zeit_ueberlappt(a, b):
    return a[0] < b[1] and b[0] < a[1]


def stoert(x, a, xb, b):
    """Stören sich Stand a (ab Meter x) und Stand b (ab Meter xb)? Zeitlich und räumlich muss es zu einer Überschneidung kommen."""
    return zeit_ueberlappt(a, b) and x < xb + b[2] and xb < x + a[2]


def einnahmen(platziert):
    """platziert: Liste von (x, anmeldung)."""
    summe = 0
    for x, a in platziert:
        summe += miete(a)
    return summe


def pruefe_markt(platziert, L):
    """Schiedsrichter: 'ok' oder der erste Verstoß."""
    for i in range(len(platziert)):
        x, a = platziert[i]
        if x < 0 or x + a[2] > L:
            return f'Stand {a} liegt nicht auf dem Markt (Meter {x} bis {x + a[2]})'
        for j in range(i):
            xb, b = platziert[j]
            if stoert(x, a, xb, b):
                return f'Stände {a} und {b} stören sich'
    return 'ok'

def erster_platz(platziert, a, L):
    kandidaten = [0]
    # dein Code: Enden der zeitlich überschneidenden Stände ergänzen
    for x in sorted(set(kandidaten)):
        # dein Code
        pass
    return None

anna = (11, 16, 5)
sophie = (16, 18, 3)
max_ = (10, 14, 4)
platziert = []
for a in (anna, sophie, max_):
    x = erster_platz(platziert, a, 1000)
    platziert.append((0 if x is None else x, a))
    print(a, 'ab Meter', x)
print(einnahmen(platziert), pruefe_markt(platziert, 1000))
print(erster_platz(platziert, (10, 14, 4), 8))

<details>
<summary>Tipp anzeigen</summary>

`for xb, b in platziert: if zeit_ueberlappt(a, b): kandidaten.append(xb + b[2])`. Danach `for x in sorted(set(kandidaten)):`. Innen: erst `if x + a[2] > L: return None`, dann alle Stände `(xb, b)` mit `stoert(x, a, xb, b)` prüfen. Ist keiner im Weg, `return x`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_markt(dateiname):
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    anmeldungen = []
    for i in range(zahlen[0]):
        anmeldungen.append((zahlen[1 + 3 * i], zahlen[2 + 3 * i], zahlen[3 + 3 * i]))
    return anmeldungen


def miete(a):
    return (a[1] - a[0]) * a[2]


def zeit_ueberlappt(a, b):
    return a[0] < b[1] and b[0] < a[1]


def stoert(x, a, xb, b):
    """Stören sich Stand a (ab Meter x) und Stand b (ab Meter xb)? Zeitlich und räumlich muss es zu einer Überschneidung kommen."""
    return zeit_ueberlappt(a, b) and x < xb + b[2] and xb < x + a[2]


def einnahmen(platziert):
    """platziert: Liste von (x, anmeldung)."""
    summe = 0
    for x, a in platziert:
        summe += miete(a)
    return summe


def pruefe_markt(platziert, L):
    """Schiedsrichter: 'ok' oder der erste Verstoß."""
    for i in range(len(platziert)):
        x, a = platziert[i]
        if x < 0 or x + a[2] > L:
            return f'Stand {a} liegt nicht auf dem Markt (Meter {x} bis {x + a[2]})'
        for j in range(i):
            xb, b = platziert[j]
            if stoert(x, a, xb, b):
                return f'Stände {a} und {b} stören sich'
    return 'ok'

def erster_platz(platziert, a, L):
    kandidaten = [0]
    for xb, b in platziert:
        if zeit_ueberlappt(a, b):
            kandidaten.append(xb + b[2])
    for x in sorted(set(kandidaten)):
        if x + a[2] > L:
            return None
        frei = True
        for xb, b in platziert:
            if stoert(x, a, xb, b):
                frei = False
                break
        if frei:
            return x
    return None

anna = (11, 16, 5)
sophie = (16, 18, 3)
max_ = (10, 14, 4)
platziert = []
for a in (anna, sophie, max_):
    x = erster_platz(platziert, a, 1000)
    platziert.append((x, a))
    print(a, 'ab Meter', x)
print(einnahmen(platziert), pruefe_markt(platziert, 1000))
print(erster_platz(platziert, (10, 14, 4), 8))
```

Das ist die Geschichte aus der Aufgabe: Anna steht bei Meter 0 bis 4, Sophie kommt erst um 16 Uhr (wenn Anna geht) und steht wieder bei Meter 0. Max (10 bis 14 Uhr) überschneidet sich zeitlich mit Anna und muss auf Meter 5 daneben. Das sind 25 + 6 + 16 = 47 Euro.
Auf einem Markt von nur 8 Metern hätte Max keinen Platz mehr (5 + 4 > 8).

</details>

#### Aufgabe E1.2

Schreibe `gierig(anmeldungen, L, regel)`: Die Anmeldungen werden **in der Reihenfolge der Schlüssel** (die Sortierzeile steht schon da) durchgegangen, und jede bekommt, wenn es geht, den Platz von `erster_platz`.
Die Funktion gibt die Liste der angenommenen Stände als `(x, anmeldung)` zurück. Danach läuft eine Tabelle für die drei echten Beispiele mit drei **Reihenfolge-Regeln**: nach Miete, nach Dauer, nach Länge.
Die Spalte **Schranke** ist die grobe obere Schranke (`schranke_grob`): Zu jeder Stunde werden höchstens `L` Meter vermietet, und nie mehr, als gewünscht wird.

**Erwartete Ausgabe:**
```
Datei            Anm.  Schranke  Miete  Dauer  Länge  beste von 3
flohmarkt4.txt     7      9395   6867   6867   6503        6867
flohmarkt5.txt    25      9927   6580   6113   6442        6580
flohmarkt6.txt     9     10000   9497  10000   9382       10000
```

In [ ]:
def lies_markt(dateiname):
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    anmeldungen = []
    for i in range(zahlen[0]):
        anmeldungen.append((zahlen[1 + 3 * i], zahlen[2 + 3 * i], zahlen[3 + 3 * i]))
    return anmeldungen


def miete(a):
    return (a[1] - a[0]) * a[2]


def zeit_ueberlappt(a, b):
    return a[0] < b[1] and b[0] < a[1]


def stoert(x, a, xb, b):
    """Stören sich Stand a (ab Meter x) und Stand b (ab Meter xb)? Zeitlich und räumlich muss es zu einer Überschneidung kommen."""
    return zeit_ueberlappt(a, b) and x < xb + b[2] and xb < x + a[2]


def einnahmen(platziert):
    """platziert: Liste von (x, anmeldung)."""
    summe = 0
    for x, a in platziert:
        summe += miete(a)
    return summe


def pruefe_markt(platziert, L):
    """Schiedsrichter: 'ok' oder der erste Verstoß."""
    for i in range(len(platziert)):
        x, a = platziert[i]
        if x < 0 or x + a[2] > L:
            return f'Stand {a} liegt nicht auf dem Markt (Meter {x} bis {x + a[2]})'
        for j in range(i):
            xb, b = platziert[j]
            if stoert(x, a, xb, b):
                return f'Stände {a} und {b} stören sich'
    return 'ok'

def erster_platz(platziert, a, L):
    kandidaten = [0]
    for xb, b in platziert:
        if zeit_ueberlappt(a, b):
            kandidaten.append(xb + b[2])
    for x in sorted(set(kandidaten)):
        if x + a[2] > L:
            return None
        frei = True
        for xb, b in platziert:
            if stoert(x, a, xb, b):
                frei = False
                break
        if frei:
            return x
    return None

def schluessel_miete(a):
    return -miete(a)                   # die teuerste Anmeldung zuerst


def schluessel_dauer(a):
    return (-(a[1] - a[0]), -a[2])     # die längste Dauer zuerst, bei gleicher Dauer der längste Stand


def schluessel_laenge(a):
    return -a[2]                       # der längste Stand zuerst

def schranke_grob(anmeldungen, L):
    """Obere Schranke: In jeder Stunde werden höchstens L Meter vermietet, aber auch nie mehr, als gewünscht wird."""
    summe = 0
    for stunde in range(8, 18):
        wunsch = 0
        for a in anmeldungen:
            if a[0] <= stunde < a[1]:
                wunsch += a[2]
        summe += min(L, wunsch)
    return summe

def gierig(anmeldungen, L, regel):
    """Reihenfolge-Regel: sortiert nach regel. Platzierungs-Regel: erster Meter, an dem der Stand passt (erster_platz).
    Gibt die Liste der angenommenen Stände als (x, anmeldung) zurück."""
    platziert = []
    reihenfolge = sorted(range(len(anmeldungen)), key=lambda i: regel(anmeldungen[i]))
    for i in reihenfolge:
        # dein Code: erster_platz fragen, bei einem Ergebnis den Stand annehmen
        pass
    return platziert

print('Datei            Anm.  Schranke  Miete  Dauer  Länge  beste von 3')
for nr in (4, 5, 6):
    anm = lies_markt(f'flohmarkt{nr}.txt')
    werte = []
    for regel in (schluessel_miete, schluessel_dauer, schluessel_laenge):
        platziert = gierig(anm, 1000, regel)
        werte.append(einnahmen(platziert))
    print(f'flohmarkt{nr}.txt {len(anm):5d} {schranke_grob(anm, 1000):9d} {werte[0]:6d} {werte[1]:6d} {werte[2]:6d} {max(werte):11d}')

<details>
<summary>Tipp anzeigen</summary>

Wie bei den Plakaten: `x = erster_platz(platziert, anmeldungen[i], L)`, und wenn `x is not None`, `platziert.append((x, anmeldungen[i]))`.

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
def lies_markt(dateiname):
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    anmeldungen = []
    for i in range(zahlen[0]):
        anmeldungen.append((zahlen[1 + 3 * i], zahlen[2 + 3 * i], zahlen[3 + 3 * i]))
    return anmeldungen


def miete(a):
    return (a[1] - a[0]) * a[2]


def zeit_ueberlappt(a, b):
    return a[0] < b[1] and b[0] < a[1]


def stoert(x, a, xb, b):
    """Stören sich Stand a (ab Meter x) und Stand b (ab Meter xb)? Zeitlich und räumlich muss es zu einer Überschneidung kommen."""
    return zeit_ueberlappt(a, b) and x < xb + b[2] and xb < x + a[2]


def einnahmen(platziert):
    """platziert: Liste von (x, anmeldung)."""
    summe = 0
    for x, a in platziert:
        summe += miete(a)
    return summe


def pruefe_markt(platziert, L):
    """Schiedsrichter: 'ok' oder der erste Verstoß."""
    for i in range(len(platziert)):
        x, a = platziert[i]
        if x < 0 or x + a[2] > L:
            return f'Stand {a} liegt nicht auf dem Markt (Meter {x} bis {x + a[2]})'
        for j in range(i):
            xb, b = platziert[j]
            if stoert(x, a, xb, b):
                return f'Stände {a} und {b} stören sich'
    return 'ok'

def erster_platz(platziert, a, L):
    kandidaten = [0]
    for xb, b in platziert:
        if zeit_ueberlappt(a, b):
            kandidaten.append(xb + b[2])
    for x in sorted(set(kandidaten)):
        if x + a[2] > L:
            return None
        frei = True
        for xb, b in platziert:
            if stoert(x, a, xb, b):
                frei = False
                break
        if frei:
            return x
    return None

def schluessel_miete(a):
    return -miete(a)                   # die teuerste Anmeldung zuerst


def schluessel_dauer(a):
    return (-(a[1] - a[0]), -a[2])     # die längste Dauer zuerst, bei gleicher Dauer der längste Stand


def schluessel_laenge(a):
    return -a[2]                       # der längste Stand zuerst

def schranke_grob(anmeldungen, L):
    """Obere Schranke: In jeder Stunde werden höchstens L Meter vermietet, aber auch nie mehr, als gewünscht wird."""
    summe = 0
    for stunde in range(8, 18):
        wunsch = 0
        for a in anmeldungen:
            if a[0] <= stunde < a[1]:
                wunsch += a[2]
        summe += min(L, wunsch)
    return summe

def gierig(anmeldungen, L, regel):
    """Reihenfolge-Regel: sortiert nach regel. Platzierungs-Regel: erster Meter, an dem der Stand passt (erster_platz).
    Gibt die Liste der angenommenen Stände als (x, anmeldung) zurück."""
    platziert = []
    reihenfolge = sorted(range(len(anmeldungen)), key=lambda i: regel(anmeldungen[i]))
    for i in reihenfolge:
        x = erster_platz(platziert, anmeldungen[i], L)
        if x is not None:
            platziert.append((x, anmeldungen[i]))
    return platziert

print('Datei            Anm.  Schranke  Miete  Dauer  Länge  beste von 3')
for nr in (4, 5, 6):
    anm = lies_markt(f'flohmarkt{nr}.txt')
    werte = []
    for regel in (schluessel_miete, schluessel_dauer, schluessel_laenge):
        platziert = gierig(anm, 1000, regel)
        assert pruefe_markt(platziert, 1000) == 'ok'
        werte.append(einnahmen(platziert))
    print(f'flohmarkt{nr}.txt {len(anm):5d} {schranke_grob(anm, 1000):9d} {werte[0]:6d} {werte[1]:6d} {werte[2]:6d} {max(werte):11d}')
```

Die Zahlen für „Miete“ und „Dauer“ stimmen mit denen des Lösungsnotebooks überein (6867, 6580, 9497 und 6867, 6113, 10000). **Es gibt keine Regel, die überall gewinnt**: Bei `flohmarkt5.txt` ist „Länge“ besser als „Dauer“, bei `flohmarkt6.txt` ist „Dauer“ perfekt (10 000 Euro, gleich der Schranke), bei `flohmarkt4.txt` sind „Miete“ und „Dauer“ gleich.

</details>

Zu den **Reihenfolge-Regeln**: „Miete“ nimmt die teuersten Anmeldungen zuerst. „Dauer“ beginnt mit den Ständen, die lange stehen, weil die am schwersten unterzubringen sind (kurze Stände füllen am Ende die Lücken). „Länge“ nimmt die breitesten Stände zuerst, die den meisten Platz brauchen.

Die Schranke ist **grob**: Sie tut so, als ließen sich Stände stundenweise zerschneiden. Nur bei `flohmarkt6.txt` ist sie genau erreichbar. Wie gut die Ergebnisse wirklich sind, wissen wir erst mit dem **Optimum**. Das schauen wir uns jetzt für **kleine** Märkte an.

#### Frage E1.3

1. Bei `flohmarkt6.txt` erreicht „Dauer“ 10 000 Euro, die Schranke. Warum ist das **bewiesen optimal**? Und was weißt du bei `flohmarkt4.txt` (Schranke 9395, beste Regel 6867)?
2. Welche Regel würdest du für **einen** Wettbewerb wählen, wenn du nur diese drei Beispiele hast? Warum ist das eine schlechte Grundlage?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Mehr als die Schranke geht nie, also ist jede Lösung mit 10 000 Euro optimal. Bei `flohmarkt4.txt` weiß man nur: Das Optimum liegt **zwischen 6867 und 9395**. Wie nah wir dran sind, ist offen. (Das Ergebnis des Solvers unten: 7370.)
2. Keine einzelne: „Miete“ ist bei `flohmarkt6.txt` schlechter als „Dauer“, und „Dauer“ ist bei `flohmarkt5.txt` schlechter als „Miete“. Drei Beispiele reichen nicht, um eine Rangfolge zu begründen. Deshalb **viele Zufallsfälle** und ein **Maßstab** (Optimum).
   Und in der 2. Runde gibt es neben den veröffentlichten Beispielen noch **weitere Eingaben, die du nicht kennst**. Die **beste von drei Regeln** abzugeben ist dort robuster als eine Regel, die zufällig bei den bekannten Beispielen gewinnt.

</details>

### Kleine Märkte: das Optimum durch Durchprobieren

Für **kleine** Märkte können wir das Optimum berechnen: Für jede Anmeldung entscheiden wir „nein“ oder „ja, ab Meter `x`“ mit **allen** möglichen `x` (Backtracking aus Notebook 2, mit einer einfachen Schranke aus Notebook 3:
Reicht der Wert aller **übrigen** Anmeldungen nicht mehr, um das bisher Beste zu schlagen, wird abgebrochen). Außerdem brauchen wir viele **Zufallsmärkte**.

In [ ]:
def lies_markt(dateiname):
    with open('daten/' + dateiname, encoding='utf-8') as datei:
        zahlen = [int(z) for z in datei.read().split()]
    anmeldungen = []
    for i in range(zahlen[0]):
        anmeldungen.append((zahlen[1 + 3 * i], zahlen[2 + 3 * i], zahlen[3 + 3 * i]))
    return anmeldungen


def miete(a):
    return (a[1] - a[0]) * a[2]


def zeit_ueberlappt(a, b):
    return a[0] < b[1] and b[0] < a[1]


def stoert(x, a, xb, b):
    """Stören sich Stand a (ab Meter x) und Stand b (ab Meter xb)? Zeitlich und räumlich muss es zu einer Überschneidung kommen."""
    return zeit_ueberlappt(a, b) and x < xb + b[2] and xb < x + a[2]


def einnahmen(platziert):
    """platziert: Liste von (x, anmeldung)."""
    summe = 0
    for x, a in platziert:
        summe += miete(a)
    return summe


def pruefe_markt(platziert, L):
    """Schiedsrichter: 'ok' oder der erste Verstoß."""
    for i in range(len(platziert)):
        x, a = platziert[i]
        if x < 0 or x + a[2] > L:
            return f'Stand {a} liegt nicht auf dem Markt (Meter {x} bis {x + a[2]})'
        for j in range(i):
            xb, b = platziert[j]
            if stoert(x, a, xb, b):
                return f'Stände {a} und {b} stören sich'
    return 'ok'

def erster_platz(platziert, a, L):
    kandidaten = [0]
    for xb, b in platziert:
        if zeit_ueberlappt(a, b):
            kandidaten.append(xb + b[2])
    for x in sorted(set(kandidaten)):
        if x + a[2] > L:
            return None
        frei = True
        for xb, b in platziert:
            if stoert(x, a, xb, b):
                frei = False
                break
        if frei:
            return x
    return None

def schluessel_miete(a):
    return -miete(a)                   # die teuerste Anmeldung zuerst


def schluessel_dauer(a):
    return (-(a[1] - a[0]), -a[2])     # die längste Dauer zuerst, bei gleicher Dauer der längste Stand


def schluessel_laenge(a):
    return -a[2]                       # der längste Stand zuerst

def gierig(anmeldungen, L, regel):
    """Reihenfolge-Regel: sortiert nach regel. Platzierungs-Regel: erster Meter, an dem der Stand passt (erster_platz).
    Gibt die Liste der angenommenen Stände als (x, anmeldung) zurück."""
    platziert = []
    reihenfolge = sorted(range(len(anmeldungen)), key=lambda i: regel(anmeldungen[i]))
    for i in reihenfolge:
        x = erster_platz(platziert, anmeldungen[i], L)
        if x is not None:
            platziert.append((x, anmeldungen[i]))
    return platziert

import random


def zufallsmarkt(startwert, n, L):
    zufall = random.Random(startwert)
    anmeldungen = []
    for i in range(n):
        von = zufall.randint(8, 16)
        bis = min(18, von + zufall.randint(1, 6))
        anmeldungen.append((von, bis, zufall.randint(5, 25)))
    return anmeldungen


def exakt_markt(anmeldungen, L, nur_kandidaten=False):
    """Das Optimum: probiert für jede Anmeldung 'nein' und jede Stelle, die passt. Von den teuersten Anmeldungen zur billigsten.
    Mit nur_kandidaten=True werden nur die Stellen 0 und 'Ende eines schon aufgestellten, zeitlich überschneidenden Standes' probiert (siehe unten)."""
    n = len(anmeldungen)
    reihenfolge = sorted(range(n), key=lambda i: -miete(anmeldungen[i]))
    rest = [0] * (n + 1)                          # rest[k]: Summe der Mieten der Anmeldungen k, k+1, ...
    for k in range(n - 1, -1, -1):
        rest[k] = rest[k + 1] + miete(anmeldungen[reihenfolge[k]])
    bestes = [0]

    def suche(k, platziert, wert):
        if wert > bestes[0]:
            bestes[0] = wert
        if k == n or wert + rest[k] <= bestes[0]:
            return
        a = anmeldungen[reihenfolge[k]]
        if nur_kandidaten:
            stellen = {0}
            for xb, b in platziert:
                if zeit_ueberlappt(a, b):
                    stellen.add(xb + b[2])
        else:
            stellen = range(L - a[2] + 1)
        for x in sorted(stellen):
            if x + a[2] <= L and all(not stoert(x, a, xb, b) for xb, b in platziert):
                suche(k + 1, platziert + [(x, a)], wert + miete(a))
        suche(k + 1, platziert, wert)

    suche(0, [], 0)
    return bestes[0]

In [ ]:
def beste_von_drei(anmeldungen, L):
    bester = 0
    for regel in (schluessel_miete, schluessel_dauer, schluessel_laenge):
        bester = max(bester, einnahmen(gierig(anmeldungen, L, regel)))
    return bester


print('Verfahren     Mittel  schlechtester Fall  optimal')
optima = [exakt_markt(zufallsmarkt(s, 8, 30), 30) for s in range(1, 41)]
for name, regel in [('Miete', schluessel_miete), ('Dauer', schluessel_dauer), ('Länge', schluessel_laenge), ('beste von 3', None)]:
    summe = 0.0
    schlechtester = 1.0
    optimal = 0
    for s in range(1, 41):
        anm = zufallsmarkt(s, 8, 30)
        if regel is None:
            wert = beste_von_drei(anm, 30)
        else:
            wert = einnahmen(gierig(anm, 30, regel))
        gute = wert / optima[s - 1]
        summe += gute
        schlechtester = min(schlechtester, gute)
        if gute == 1:
            optimal += 1
    print(f'{name:11} {100 * summe / 40:7.1f} % {100 * schlechtester:13.1f} % {optimal:6d} von 40')

Verfahren     Mittel  schlechtester Fall  optimal
Miete          96.6 %          80.5 %     23 von 40
Dauer          92.9 %          60.8 %     21 von 40
Länge          87.1 %          54.1 %      9 von 40
beste von 3    97.7 %          82.6 %     28 von 40


#### Frage E1.4

1. Lies die Tabelle über die 40 kleinen Märkte (8 Anmeldungen, Markt von 30 Metern). Welche Reihenfolge-Regel ist die beste, welche die schlechteste, und wie viel bringt „beste von 3“?
2. Auf den drei echten Beispielen hat „Dauer“ bei `flohmarkt6.txt` gewonnen, „Miete“ bei `flohmarkt5.txt`, und bei `flohmarkt4.txt` waren beide gleich. Passt das zu dieser Tabelle? Was ist die Lehre?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Im Mittel ist „Miete“ mit 96,6 % am besten, danach „Dauer“ (92,9 %) und „Länge“ (87,1 %). „Beste von 3“ erreicht 97,7 % (schlechtester Fall 82,6 %, optimal in 28 von 40 Fällen).
   Der Gewinn entsteht, weil die Regeln **verschiedene Fälle** gut lösen.
2. Ja, „Miete“ gewinnt im Mittel, aber **nicht überall** („Dauer“ gewinnt bei `flohmarkt6.txt`). Das ist die Lehre dieses Notebooks: Die beste einzelne Regel im Mittel ist nicht die beste auf jeder Eingabe, deshalb **kombiniert** man und **misst** man an vielen Fällen.

</details>

### Finde den Fehler: Kandidatenstellen auch beim exakten Suchen?

Beim Gierigen sind die Kandidatenstellen **richtig** (der erste freie Meter liegt immer bei 0 oder am Ende eines Standes, siehe Abschnitt 4). Jemand benutzt dieselbe Idee im **exakten** Verfahren (`nur_kandidaten=True`) und
vergleicht mit dem vollen Durchprobieren an einem winzigen Markt (6 Anmeldungen, 30 Meter, Startwert 27):

In [ ]:
anm = zufallsmarkt(27, 6, 30)
print(anm)
print('alle Stellen probiert:', exakt_markt(anm, 30))
print('nur Kandidatenstellen:', exakt_markt(anm, 30, True))

[(15, 18, 13), (12, 14, 7), (9, 12, 22), (13, 16, 16), (14, 16, 12), (11, 15, 7)]
alle Stellen probiert: 195
nur Kandidatenstellen: 181


#### Frage E1.5

1. Warum findet das exakte Verfahren mit Kandidatenstellen nicht das Optimum, obwohl es beim gierigen Verfahren immer die richtige Stelle findet?
2. Was gilt sinngemäß für **jedes** Verfahren, das das Optimum findet und dabei Stellen weglässt?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. Beim Gierigen ist die Regel **„die erste freie Stelle“**, und die liegt immer bei 0 oder am Ende eines schon aufgestellten Standes. Beim exakten Verfahren wird auch **jede andere** Stelle gebraucht: Die Suche stellt die Anmeldungen in der Reihenfolge der Mieten auf, nicht von links nach rechts. Ein später aufgestellter Stand kann am besten in eine Lücke **links von** einem schon aufgestellten Stand passen.
   Die Kandidaten sind nur Enden **schon aufgestellter** Stände, eine solche Lücke fehlt unter ihnen. In dem Beispiel fehlen deshalb 14 Euro (195 statt 181).
2. Das Weglassen von Stellen **ist nur dann ungefährlich, wenn man begründen kann**, dass die gesuchte Stelle immer unter den übrigen ist (beim Gierigen: die erste freie Stelle). Sonst wird aus dem „exakten“ Verfahren stillschweigend eine **Heuristik**. Solche Fehler findet man nur durch den **Vergleich mit dem vollen Durchprobieren** an kleinen Fällen, wie hier.

</details>

## Zum Schluss: Die Güte mit MiniZinc messen

Für Märkte mit mehr als 8 Anmeldungen ist das Durchprobieren in Python zu langsam. Ein **Solver** liefert das Optimum auch dort, und damit können wir die Güte der gierigen Regeln messen. Du kennst von den MiniZinc-Arbeitsblättern `forall`,
`sum` mit `where`, Variablen als Index und `.dzn`-Dateien. Neu sind hier die logischen Verknüpfungen: `/\` ist „und“, `\/` ist „oder“, `A -> B` heißt „wenn A, dann B“. Ein `var bool` zählt in einer Summe als 0 oder 1.
**MiniZinc läuft nicht in JupyterLite:** Kopiere das Modell in die MiniZinc IDE oder in den [MiniZinc Playground](https://play.minizinc.org/) und füge die Daten als `.dzn` dazu (die Python-Zelle darunter erzeugt sie).

```minizinc
int: n;                                    % Zahl der Anmeldungen
int: L;                                    % Länge des Marktes in Metern
array[1..n] of int: von;
array[1..n] of int: bis;
array[1..n] of int: laenge;

array[1..n] of var bool: k;                % Anmeldung i angenommen?
array[1..n] of var 0..L: x;                % erster Meter des Standes i
constraint forall(i in 1..n)(x[i] + laenge[i] <= L);

% zwei angenommene Stände, die sich zeitlich überschneiden, stehen nebeneinander
constraint forall(i, j in 1..n where i < j /\ von[i] < bis[j] /\ von[j] < bis[i])(
  (k[i] /\ k[j]) -> (x[i] + laenge[i] <= x[j] \/ x[j] + laenge[j] <= x[i]));

var int: einnahmen = sum(i in 1..n)(k[i] * (bis[i] - von[i]) * laenge[i]);
solve maximize einnahmen;
output ["einnahmen = \(einnahmen)\n", "angenommen: \([i | i in 1..n where fix(k[i])])\n"];
```

Die Bedingung schreiben wir nur für Paare auf, die sich zeitlich überschneiden (`where` mit festen Daten). Für alle anderen Paare gibt es nichts zu prüfen. `fix(k[i])` in der Ausgabe liest den Wert einer Variablen in der Lösung.
Der Solver sucht **gleichzeitig** die Auswahl **und** alle Standorte. Das ist die Aufgabe ohne jede Heuristik, ganz wie die Formalisierung der Aufgabe.

In [ ]:
def als_dzn(anmeldungen, L):
    von = [a[0] for a in anmeldungen]
    bis = [a[1] for a in anmeldungen]
    laenge = [a[2] for a in anmeldungen]
    return f'n = {len(anmeldungen)};\nL = {L};\nvon = {von};\nbis = {bis};\nlaenge = {laenge};'


print(als_dzn(lies_markt('flohmarkt4.txt'), 1000))

n = 7;
L = 1000;
von = [8, 15, 12, 8, 9, 8, 14];
bis = [15, 18, 15, 9, 15, 12, 18];
laenge = [249, 526, 520, 503, 477, 171, 401];


Die Ausgabe für `flohmarkt5.txt` (`==========` heißt: Optimum bewiesen):

**Ausgabe:**

```
einnahmen = 8705
angenommen: [2, 3, 6, 7, 9, 14, 19, 20]
----------
==========
```

Chuffed ist bei diesen drei echten Beispielen sofort fertig. Jetzt können wir die gierigen Regeln **gegen das Optimum** messen:

| Beispiel | Schranke | beste von 3 | Optimum (MiniZinc) | Güte von „beste von 3“ |
|---|---|---|---|---|
| `flohmarkt4.txt` | 9395 | 6867 | 7370 | 93,2 % |
| `flohmarkt5.txt` | 9927 | 6580 | 8705 | 75,6 % |
| `flohmarkt6.txt` | 10000 | 10000 | 10000 | 100,0 % |

Bei `flohmarkt4.txt` und `flohmarkt5.txt` liegt die Schranke (9395 und 9927) weit über dem Optimum (7370 und 8705). Erst das Optimum zeigt, wie gut die Regeln wirklich sind: 93,2 % und 75,6 %. Bei `flohmarkt5.txt` fehlt uns also fast ein Viertel: Die gierigen Regeln scheitern an den wenigen, sehr langen Ständen.
Das passt zu Notebook 04: Hier hilft **lokale Suche** (zufällige Reihenfolgen, Stände wegnehmen und neu füllen), wie im Lösungsnotebook gezeigt (dort 8705 mit 2000 zufälligen Reihenfolgen, Versuch 1333).

**Probe:** Für die 40 kleinen Märkte `zufallsmarkt(startwert, 8, 30)` mit Startwert 1 bis 40 liefert MiniZinc dasselbe Optimum wie `exakt_markt` in Python (beim Erzeugen dieses Notebooks geprüft).
So kann man dem Solver auch bei den Fällen trauen, für die man ihn braucht, weil das Durchprobieren nicht mehr geht.

**Mittelgroße Märkte:** 20 Anmeldungen auf 60 Metern, `zufallsmarkt(startwert, 20, 60)`. Python-Durchprobieren wäre hier schon sehr langsam, Chuffed ist in jedem der zwölf Fälle schnell fertig (gemessen beim Erzeugen):

| Startwert | Miete | Dauer | Länge | beste von 3 | Optimum (MiniZinc) | Güte |
|---|---|---|---|---|---|---|
| 1 | 441 | 397 | 382 | 441 | 441 | 100,0 % |
| 2 | 453 | 445 | 367 | 453 | 475 | 95,4 % |
| 3 | 407 | 405 | 418 | 418 | 485 | 86,2 % |
| 4 | 516 | 460 | 518 | 518 | 537 | 96,5 % |
| 5 | 385 | 385 | 376 | 385 | 420 | 91,7 % |
| 6 | 426 | 426 | 389 | 426 | 479 | 88,9 % |
| 7 | 468 | 468 | 397 | 468 | 527 | 88,8 % |
| 8 | 452 | 422 | 434 | 452 | 500 | 90,4 % |
| 9 | 471 | 443 | 451 | 471 | 531 | 88,7 % |
| 10 | 484 | 430 | 467 | 484 | 529 | 91,5 % |
| 11 | 523 | 507 | 473 | 523 | 532 | 98,3 % |
| 12 | 458 | 395 | 502 | 502 | 551 | 91,1 % |

Im Mittel erreicht „beste von 3“ 92,3 % des Optimums, im schlechtesten Fall 86,2 %, und in 1 von 12 Fällen ist sie optimal.
**Das ist die ehrliche Aussage, die in eine Dokumentation gehört:** „Gemessen an 12 Märkten mit 20 Anmeldungen: im Mittel 92,3 % des Optimums, im schlechtesten Fall 86,2 %.“ Für große Märkte (500 Anmeldungen) kennen wir das Optimum nicht, dort bleibt nur die Schranke.

**Grenzen des Solvers.** Bei 30 Anmeldungen auf 100 Metern (`zufallsmarkt(startwert, 30, 100)`) war Chuffed bei den ausprobierten Startwerten 1, 2 und 3 nach 30 Sekunden **nicht fertig** und hatte nicht bewiesen, dass er das Optimum gefunden hat
(gemessen beim Entwickeln, die genaue Dauer hängt vom Rechner ab). Die 490 bis 735 Anmeldungen der großen Beispiele sind gar nicht zu schaffen (mehr als 90 000 Paarbedingungen; siehe das Lösungsnotebook).
Man setzt dem Solver deshalb **immer ein Zeitlimit** und schreibt ehrlich, was er in der Zeit **bewiesen** hat und was **offen** blieb. Für große Fälle braucht man die gierigen Regeln und die Schranke.

## Wettbewerbs-Ecke 2 (für Schnelle): Die Siedler

*Angelehnt an Aufgabe 3 „Die Siedler“ der 2. Runde des 42. Bundeswettbewerbs Informatik (2023/24).*

In einem Gebiet sollen möglichst viele **Ortschaften** gegründet werden, die **paarweise mindestens 10 km** auseinanderliegen. Im Wettbewerb ist das Gebiet ein beliebiges Polygon, es gibt ein Gesundheitszentrum und je nach Schutz verschiedene Mindestabstände.
Wir vereinfachen stark: Das Gebiet ist ein **Rechteck** von `W` mal `H` Kilometern (Ortschaften dürfen auch auf dem Rand liegen), und es gilt nur der Mindestabstand 10 km. Das Lösungsnotebook [Die Siedler](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/42_DieSiedler/siedler_loesung.ipynb) macht die ganze Aufgabe.

Hier gibt es **unendlich viele** Stellen. Ein Raster von Punkten wäre eine grobe Näherung und würde bei Abstand 0,5 km für ein Gebiet von 55 mal 37 km für jede Ortschaft 111 · 75 = 8 325 Punkte prüfen. **Kandidatenstellen** machen es genau und sparsam:
Eine Ortschaft, die so **eng** wie möglich steht, berührt den Mindestabstand zu **zwei** anderen Ortschaften oder zu **einer** Ortschaft und dem **Rand**. Sie steht also auf dem **Schnittpunkt zweier Kreise** (Radius 10 km um zwei Ortschaften)
oder auf dem **Schnittpunkt eines Kreises mit einer Randgeraden**. Die Reihenfolge-Regel ist hier die **Reihenfolge der Kandidaten**: Der Kandidat mit dem **kleinsten Schlüssel** kommt zuerst dran, und wir verwalten sie in einer **Prioritätswarteschlange** (`heapq`, wie in Notebook 7 von learn_bwinf1).

#### Aufgabe E2.1

Schreibe `setze_orte(W, H, schluessel)`. Die Schlange `schlange` enthält Paare `(schluessel(p), p)` und beginnt mit dem Kandidaten `(0.0, 0.0)` (die Ecke). Solange die Schlange nicht leer ist:

1. Hole mit `heapq.heappop` den Kandidaten `p` mit dem kleinsten Schlüssel und zähle ihn bei `geprueft`.
2. Liegt `p` **nicht** im Gebiet oder ist es **nicht weit genug** von allen bisherigen Orten (`im_gebiet`, `weit_genug`), geht es mit dem nächsten Kandidaten weiter (`continue`).
3. Sonst entstehen **neue Kandidaten**: `kreis_kreis(o, p, ABSTAND)` für **jede** bisherige Ortschaft `o` und `kreis_rand(p, ABSTAND, W, H)`. Lege jeden neuen Kandidaten, der im Gebiet liegt, mit `heapq.heappush` in die Schlange.
   Danach wird `p` eine Ortschaft (`orte.append(p)`).

Die Funktion gibt `(orte, geprueft)` zurück. Der Schiedsrichter `pruefe_orte` prüft das Ergebnis.

**Erwartete Ausgabe:**
```
30 Ortschaften, 380 Kandidaten geprüft, ok
```

In [ ]:
import math
import heapq

ABSTAND = 10.0          # Mindestabstand zweier Ortschaften in km
TOL = 1e-7              # Toleranz gegen Rundungsfehler bei Kommazahlen


def kreis_kreis(m1, m2, r):
    """Schnittpunkte zweier Kreise mit Radius r um m1 und m2 (Liste mit 0 oder 2 Punkten)."""
    d = math.hypot(m2[0] - m1[0], m2[1] - m1[1])
    if d == 0 or d > 2 * r:
        return []
    h = math.sqrt(max(0.0, r * r - d * d / 4))
    mx = (m1[0] + m2[0]) / 2
    my = (m1[1] + m2[1]) / 2
    sx = (m2[1] - m1[1]) / d            # Einheitsvektor senkrecht zur Verbindung
    sy = -(m2[0] - m1[0]) / d
    return [(mx + h * sx, my + h * sy), (mx - h * sx, my - h * sy)]


def kreis_rand(m, r, W, H):
    """Schnittpunkte des Kreises mit Radius r um m mit den vier Randgeraden des Rechtecks (Punkte außerhalb sortiert später das Prüfen aus)."""
    punkte = []
    for x in (0, W):
        dx = abs(x - m[0])
        if dx <= r:
            dy = math.sqrt(r * r - dx * dx)
            punkte += [(x, m[1] + dy), (x, m[1] - dy)]
    for y in (0, H):
        dy = abs(y - m[1])
        if dy <= r:
            dx = math.sqrt(r * r - dy * dy)
            punkte += [(m[0] + dx, y), (m[0] - dx, y)]
    return punkte


def im_gebiet(p, W, H):
    return -TOL <= p[0] <= W + TOL and -TOL <= p[1] <= H + TOL


def weit_genug(p, orte):
    for o in orte:
        if math.hypot(p[0] - o[0], p[1] - o[1]) < ABSTAND - TOL:
            return False
    return True


def pruefe_orte(orte, W, H):
    """Schiedsrichter: alle Paare probieren."""
    for i in range(len(orte)):
        if not im_gebiet(orte[i], W, H):
            return f'Ort {i} liegt außerhalb'
        for j in range(i):
            if math.hypot(orte[i][0] - orte[j][0], orte[i][1] - orte[j][1]) < ABSTAND - TOL:
                return f'Orte {j} und {i} liegen zu dicht beieinander'
    return 'ok'


def schluessel_unten_links(p):
    return (round(p[1], 6), p[0])       # erst die kleinste Höhe y, dann das kleinste x (gerundet gegen Rundungsrauschen)


def schluessel_links_unten(p):
    return (round(p[0], 6), p[1])

def setze_orte(W, H, schluessel):
    orte = []
    start = (0.0, 0.0)
    schlange = [(schluessel(start), start)]
    geprueft = 0
    while len(schlange) > 0:
        # dein Code
        break
    return orte, geprueft

orte, geprueft = setze_orte(55, 37, schluessel_unten_links)
print(len(orte), 'Ortschaften,', geprueft, 'Kandidaten geprüft,', pruefe_orte(orte, 55, 37))

<details>
<summary>Tipp anzeigen</summary>

`_, p = heapq.heappop(schlange)`. Dann `if not im_gebiet(p, W, H) or not weit_genug(p, orte): continue`. Dann `neue = kreis_rand(p, ABSTAND, W, H)` und für jedes `o` in `orte` `neue += kreis_kreis(o, p, ABSTAND)`.
Für jedes `q` in `neue` mit `im_gebiet(q, W, H)` `heapq.heappush(schlange, (schluessel(q), q))`. Zuletzt `orte.append(p)`. Wichtig: Die neuen Kandidaten entstehen **bevor** `p` zu `orte` kommt (sonst wäre `kreis_kreis(p, p, ...)` dabei, das liefert aber eine leere Liste, weil der Abstand 0 ist).

</details>

<details>
<summary>Lösung anzeigen (erst selbst probieren!)</summary>

```python
import math
import heapq

ABSTAND = 10.0          # Mindestabstand zweier Ortschaften in km
TOL = 1e-7              # Toleranz gegen Rundungsfehler bei Kommazahlen


def kreis_kreis(m1, m2, r):
    """Schnittpunkte zweier Kreise mit Radius r um m1 und m2 (Liste mit 0 oder 2 Punkten)."""
    d = math.hypot(m2[0] - m1[0], m2[1] - m1[1])
    if d == 0 or d > 2 * r:
        return []
    h = math.sqrt(max(0.0, r * r - d * d / 4))
    mx = (m1[0] + m2[0]) / 2
    my = (m1[1] + m2[1]) / 2
    sx = (m2[1] - m1[1]) / d            # Einheitsvektor senkrecht zur Verbindung
    sy = -(m2[0] - m1[0]) / d
    return [(mx + h * sx, my + h * sy), (mx - h * sx, my - h * sy)]


def kreis_rand(m, r, W, H):
    """Schnittpunkte des Kreises mit Radius r um m mit den vier Randgeraden des Rechtecks (Punkte außerhalb sortiert später das Prüfen aus)."""
    punkte = []
    for x in (0, W):
        dx = abs(x - m[0])
        if dx <= r:
            dy = math.sqrt(r * r - dx * dx)
            punkte += [(x, m[1] + dy), (x, m[1] - dy)]
    for y in (0, H):
        dy = abs(y - m[1])
        if dy <= r:
            dx = math.sqrt(r * r - dy * dy)
            punkte += [(m[0] + dx, y), (m[0] - dx, y)]
    return punkte


def im_gebiet(p, W, H):
    return -TOL <= p[0] <= W + TOL and -TOL <= p[1] <= H + TOL


def weit_genug(p, orte):
    for o in orte:
        if math.hypot(p[0] - o[0], p[1] - o[1]) < ABSTAND - TOL:
            return False
    return True


def pruefe_orte(orte, W, H):
    """Schiedsrichter: alle Paare probieren."""
    for i in range(len(orte)):
        if not im_gebiet(orte[i], W, H):
            return f'Ort {i} liegt außerhalb'
        for j in range(i):
            if math.hypot(orte[i][0] - orte[j][0], orte[i][1] - orte[j][1]) < ABSTAND - TOL:
                return f'Orte {j} und {i} liegen zu dicht beieinander'
    return 'ok'


def schluessel_unten_links(p):
    return (round(p[1], 6), p[0])       # erst die kleinste Höhe y, dann das kleinste x (gerundet gegen Rundungsrauschen)


def schluessel_links_unten(p):
    return (round(p[0], 6), p[1])

def setze_orte(W, H, schluessel):
    orte = []
    start = (0.0, 0.0)
    schlange = [(schluessel(start), start)]
    geprueft = 0
    while len(schlange) > 0:
        _, p = heapq.heappop(schlange)
        geprueft += 1
        if not im_gebiet(p, W, H) or not weit_genug(p, orte):
            continue
        neue = kreis_rand(p, ABSTAND, W, H)
        for o in orte:
            neue += kreis_kreis(o, p, ABSTAND)
        for q in neue:
            if im_gebiet(q, W, H):
                heapq.heappush(schlange, (schluessel(q), q))
        orte.append(p)
    return orte, geprueft

orte, geprueft = setze_orte(55, 37, schluessel_unten_links)
print(len(orte), 'Ortschaften,', geprueft, 'Kandidaten geprüft,', pruefe_orte(orte, 55, 37))
```

Die Schlange enthält auch Kandidaten, die später **nicht mehr** weit genug entfernt sind (andere Ortschaften sind inzwischen dazugekommen): Deshalb wird **beim Herausnehmen** noch einmal geprüft. Das ist ein Muster, das man bei Warteschlangen-Verfahren oft braucht.

</details>

In [ ]:
def schranke_orte(W, H):
    """Obere Schranke: dichteste Packung (Sechsecke); am Rand darf man 5 km über den Rand 'hinaus' rechnen."""
    return (W * H + 2 * (W + H) * ABSTAND / 2) / (math.sqrt(3) / 2 * ABSTAND ** 2)


print('Gebiet      Schranke  Quadratraster  unten-links  links-unten')
for W, H in [(55, 37), (60, 40), (100, 60)]:
    raster = (int(W // ABSTAND) + 1) * (int(H // ABSTAND) + 1)
    a = len(setze_orte(W, H, schluessel_unten_links)[0])
    b = len(setze_orte(W, H, schluessel_links_unten)[0])
    print(f'{W:4d} x {H:3d} {schranke_orte(W, H):9.1f} {raster:14d} {a:12d} {b:12d}')

Gebiet      Schranke  Quadratraster  unten-links  links-unten
  55 x  37      34.1             24           30           28
  60 x  40      39.3             35           33           32
 100 x  60      87.8             77           74           78


#### Frage E2.2

1. Wie nah kommen die Pläne an die Schranke (dichteste Packung, `55 x 37` hat die Schranke 34,1)? Und wie viel besser als das einfache Quadratraster mit Abstand 10 km (jede Ortschaft in einem Gitter mit 10 km Abstand)?
2. Welche der beiden Reihenfolgen der Kandidaten ist besser? Reicht das für eine Aussage?
3. Das Programm prüft für 55 mal 37 km nur 380 Kandidaten. Wie viele Punkte hätte ein Raster mit Abstand 0,5 km **je Ortschaft** geprüft, und warum würde es das Optimum trotzdem nicht sicher treffen?

✏️ *Deine Antwort (Doppelklick zum Bearbeiten):*

<details>
<summary>Lösung anzeigen (erst selbst überlegen!)</summary>

1. `55 x 37`: 30 von höchstens 34 Ortschaften, also 88 % der Schranke. Das Quadratraster schafft nur 24. Bei `60 x 40` ist das Quadratraster (35 Punkte) **besser** als unser Verfahren (33), weil sich 60 und 40 durch 10 teilen und das Raster dort perfekt aufgeht.
   Bei `100 x 60` schlägt das Raster (77) die Reihenfolge „unten-links“ (74), „links-unten“ (78) liegt knapp darüber. **Die Heuristik ist nicht in jedem Fall besser als eine einfache Idee.** Das merkt man nur, wenn man mehrere Fälle misst und eine einfache Vergleichslösung danebenstellt.
2. Mal ist die eine besser, mal die andere (30 gegen 28, 33 gegen 32, 74 gegen 78). Wie in Abschnitt 3 liegt es nahe, **beide** zu probieren und die bessere zu nehmen (doppelte Arbeit, wie gehabt).
3. 111 mal 75 = 8 325 Punkte je Ortschaft, also rund 250 000 für 30 Ortschaften, gegenüber 380. Die engsten Stellen liegen auf **Schnittpunkten** (Kreis mit Kreis), die fast nie genau auf einem Rasterpunkt liegen. Das Raster würde also mehr Abstand lassen und weniger Ortschaften unterbringen.
   MiniZinc passt hier nicht: Die Koordinaten sind Kommazahlen, die Abstände enthalten Wurzeln, und die Solver aus den Arbeitsblättern rechnen mit ganzen Zahlen.

</details>

## Das hast du gelernt

```python
# Konstruktive Heuristik = Reihenfolge-Regel + Platzierungs-Regel
def baue(eingabe, regel):                  # regel: Funktion, kleinster Schlüssel zuerst
    reihenfolge = sorted(range(len(eingabe)), key=lambda i: regel(eingabe[i]))
    loesung = []
    for i in reihenfolge:
        stelle = erste_passende_stelle(loesung, eingabe[i])      # Kandidatenstellen: nur 0 und Kanten bereits gestellter Teile
        if stelle is not None:
            loesung.append((stelle, eingabe[i]))
    return loesung

# mehrere Regeln: das beste von k (k-fache Arbeit)
bester = max(wert(baue(eingabe, regel)) for regel in [regel_a, regel_b, regel_c])
```

- Eine **konstruktive Heuristik** baut die Lösung Schritt für Schritt auf und nimmt nichts zurück. Man **trennt** die **Reihenfolge-Regel** (wer kommt zuerst dran?) von der **Platzierungs-Regel** (wohin?) und übergibt die Reihenfolge-Regel als **Funktion**.
- Es gibt **meist keinen Beweis**, dass eine gierige Regel optimal ist. Man **entwirft mehrere Regeln** und **misst**: an vielen Zufallsfällen mit festem Startwert, als **Tabelle** mit Mittelwert, **schlechtestem Fall** und der Zahl der **optimalen** Fälle, gegen das **Optimum** kleiner Fälle (Brute Force, Rucksack-Tabelle, MiniZinc) oder gegen eine **Schranke**.
- Zu jeder Regel gibt es ein **Gegenbeispiel**. Man findet es **automatisch**, indem man kleine Zufallsfälle gegen Brute Force prüft. Es zeigt die **Schwäche** der Regel (blockierter Platz, Kleinkram, Entscheidung ohne Rücksicht auf den Rest).
- **Kandidatenstellen** statt aller Stellen: Die erste freie Stelle liegt bei 0 oder an einer Kante eines schon gestellten Teils (Plakate, Stände), die engste Stelle auf einem Kreisschnittpunkt (Siedler). Das ist **sparsam und trotzdem genau**, solange man begründen kann, dass das Optimum **dieser Regel** darunter liegt.
  Beim **exakten** Suchen darf man Stellen **nicht** einfach weglassen (Finde den Fehler in der Flohmarkt-Ecke).
- **Regeln kombinieren** („beste von k“) ist nie schlechter als jede einzelne Regel, kostet aber **k-mal** so viel Rechenzeit. Zufallsregeln bringen wenig. Das gehört **ehrlich** in die Dokumentation.
- Ein **Solver** (MiniZinc) gibt für mittlere Größen das Optimum und macht damit die **Güte** der Regeln messbar. Bei großen Fällen wird er nicht fertig, dort bleibt nur die Schranke.

### Aufgaben der 2. Runde dazu

Bearbeite eine Aufgabe zuerst selbst und sieh dir das Lösungsnotebook erst danach an.

- [Flohmarkt in Langdorf](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/39_FlohmarktInLangdorf/flohmarkt_loesung.ipynb) (39. BwInf): Reihenfolgen vergleichen, zufällige Reihenfolgen, Räumen und Neufüllen, lückenlose Belegung, Schranke mit MiniZinc
- [Die Siedler](https://ktheu.github.io/bwinfN/notebooks/index.html?path=bwinf2/42_DieSiedler/siedler_loesung.ipynb) (42.): Ortschaften gierig setzen, Kandidaten statt Suche im Gebiet, obere Schranke

Weitere gierige Aufgaben stehen in der Werkzeugübersicht der 2. Runde (Abschnitt „Heuristiken“). Im nächsten Notebook (04) geht es um **lokale Suche**: Du nimmst die gierige Lösung als Start und verbesserst sie durch viele kleine Änderungen.